#Qué se ha hecho:
- Fase 1. Diagnóstico inicial de los datos
- Carga y exploración del archivo BBDD_sintetico_consistente.xlsx.
- Análisis de dimensiones, tipos de datos, valores nulos, duplicados y cardinalidad.
- Identificación de las principales variables de clientes, contratos y productos.
- Fase 2. Análisis de calidad de los datos
- Detección de valores ausentes, guiones utilizados como valores especiales y problemas de codificación de caracteres.
- Revisión de estados contractuales, fechas y categorías.
- Identificación de posibles inconsistencias en los identificadores y atributos de clientes.
- Fase 3. Análisis de clientes, contratos y productos
- Identificación de clientes y contratos únicos.
- Clasificación de clientes en tres grupos: solo electricidad, solo gas y ambos productos.
- Detección de contratos asociados a más de un identificador de cliente.
- Identificación de clientes con valores contradictorios en Tipo Cliente.
- Fase 4. Análisis de calidad y criterios de consolidación
- Revisión de las inconsistencias detectadas.
- Definición de criterios para conservar los registros sin fusionar automáticamente contratos potencialmente conflictivos.
- Evaluación de variables candidatas para caracterizar a los clientes.
- Fase 5. Consolidación y creación de variables
- Construcción de una base con una fila por cliente.
- Creación de indicadores de tenencia de electricidad y gas.
- Cálculo del número de productos, contratos y conflictos en Tipo Cliente.
- Cálculo de la fecha de primera creación registrada y de la antigüedad en meses.
- Análisis de la diversidad de potencias eléctricas y escalones de consumo de gas.
- Incorporación de Num_Potencias_Distintas y Num_Escalones_Gas_Distintos.
- Validación de las dimensiones, unicidad de la clave y consistencia de los principales recuentos.
- Fase 6.1. Diagnóstico de variables candidatas
- Análisis descriptivo de las variables de cliente.
- Estudio de distribuciones, valores extremos y ceros estructurales.
- Identificación de redundancia entre Tiene_Luz, Tiene_Gas, Num_Productos y Clasificacion_Suministro.
- Evaluación preliminar de correlaciones entre variables numéricas.
- Propuesta de dos alternativas de variables para la segmentación.
- Fase 6.2. Preparación metodológica para el modelado
- Estudio de la asimetría de Num_Contratos y evaluación de una posible transformación logarítmica.
- Revisión de la codificación One-Hot de la tipología de suministro.
- Análisis de los efectos de los ceros estructurales en las variables de variabilidad técnica.
- Evaluación metodológica preliminar de StandardScaler y RobustScaler.
- Definición de controles para preservar la integridad de los datos durante las transformaciones.


### 1. Carga del Fichero y Estructura General
En esta celda procedemos a importar las librerías necesarias y a cargar el archivo `BBDD_sintetico_consistente.xlsx` para conocer sus dimensiones, las primeras filas y los tipos de datos asignados automáticamente.

In [1]:
import pandas as pd
import numpy as np

# Ruta del archivo Excel identificado en el entorno
file_path = '/content/BBDD_sintetico_consistente.xlsx'

# Carga del fichero sin modificar el contenido original
df = pd.read_excel(file_path)

# 1. Número de filas y columnas
print(f"Dimensiones del dataset: {df.shape[0]} filas y {df.shape[1]} columnas.\n")

# 2. Nombres de todas las columnas
print("Nombres de las columnas del dataset:")
print(list(df.columns), "\n")

# 3. Primeras cinco filas
print("Primeras 5 filas del dataset:")
display(df.head(5))

# 4. Tipos de datos de las columnas
print("\nTipos de datos de cada columna:")
display(df.dtypes)

Dimensiones del dataset: 664371 filas y 36 columnas.

Nombres de las columnas del dataset:
['Tipo Cliente', 'Contrato', 'NumSeq', 'Est Contrato', 'Tarifa', 'CUI', 'CPE', 'Origem', 'NOA', 'NIF', 'Num Cliente', 'Modo Envio Fact', 'DT ActivaÃ§Ã£o', 'DT Mod', 'DT Bax', 'Data CriaÃ§Ã£o', 'Prod', 'TP Conta', 'Ciclo Hr', 'Distrito', 'Concelho', 'Freguesia', 'Email', 'CAE', 'Potencia', 'EscalÃ£o Consumo', 'nome', 'Canal Entrada', 'Sub Canal Entrada', 'Nome Cliente', 'Morada Envio', 'CodigoPostal', 'LocalPostal', 'Tipo de ComunicaÃ§Ã£o', 'Identificador Parceria', 'Campanha'] 

Primeras 5 filas del dataset:


,Tipo Cliente,Contrato,NumSeq,Est Contrato,Tarifa,CUI,CPE,Origem,NOA,NIF,...,nome,Canal Entrada,Sub Canal Entrada,Nome Cliente,Morada Envio,CodigoPostal,LocalPostal,Tipo de ComunicaÃ§Ã£o,Identificador Parceria,Campanha
0,91 - Domesticos,803988102,102,14 - Contrato Activado nÃ£o Factur.,+ p/Si GÃ¡s 2506,1c170b569932410bb0ed9d19273d298e,d9d117f973fc45b8be71298fefbcb138,CAT RECECAO CGI,N,49c95b8bd62443dea9530dae6fe292ab,...,-,CANAL UNICO,CAT RECECAO CGI,6d0c69c261e04d35a69022b5913beabe,ea74777cb79a43f0a34dc6c010b7e22c,e7f448200be742acbcbcbdf72291fea4,FIGUEIRA DA FOZ,-,-,-
1,91 - Domesticos,206227101,101,14 - Contrato Activado nÃ£o Factur.,+ p/Si GÃ¡s 2606,869651e2927649638b07f4fc7b9c13b8,ccb05374d2124e84bc22e1e061b1a6f3,CAT Emissao,N,c3de3fd97bae4ccfac59e4e36d369178,...,-,Teleperformance,CAT Emissao,068a6309a8004c1aa12778daf929f005,6bd269d160dd44da823e1c09e7be36bd,b0c9ce220b13458aa8d136cd56f94098,GUARDA,-,297500,MAIS PARA SI 15%
2,91 - Domesticos,387226103,103,14 - Contrato Activado nÃ£o Factur.,+ p/Si GÃ¡s 2606,61d64e5682b644f7ad1f4ef5e629e92e,bbdc2f80966e46489491aa0915b46f33,GoConnection,N,d9ce40a7ddc44b34b536ce105e2a206d,...,-,Go Connection,CAT Emissao,6a3fa58efd8345c2bbc6973a8b3a65ca,92fceff00a4e423d8b55606bebba7dbd,3255bbe5c0814666a75a96ec6dadc445,PAÃ‡O DE ARCOS,-,1432665,RECUPERACAO
3,91 - Domesticos,413113102,102,14 - Contrato Activado nÃ£o Factur.,+ p/Si GÃ¡s 2606,ec8b1ea8d3a44fed9bfc705046dd7fbb,d0855b4779b040268d81fd89bc45d174,CAT RECECAO CGI,N,4b6bef9c2483479888a39b3243d9a003,...,-,CANAL UNICO,CAT RECECAO CGI,cfbd624519304c7791eaccac7fc9b87e,ccc7ae4a15bf483780bf06ce7770f511,925a3d7230274516b32bedc9c0eadc41,AMADORA,-,-,-
4,91 - Domesticos,418340104,104,14 - Contrato Activado nÃ£o Factur.,+ p/Si GÃ¡s 2606,54b35d02494b43c291c91bb5972050bf,3077a60e9266487688f78e5ffe01f362,GoConnection,N,f58e5f45305a41788ab504a1faed9d47,...,-,Go Connection,CAT Emissao,4454eb3bce1544a38d5effd39f2387e3,9ac952d779704b228551496a48f8b4e5,05f54eda3e79403c91e2f99c3c4d63ce,ERMESINDE,-,1442282,RECUPERACAO



Tipos de datos de cada columna:


,0
Tipo Cliente,object
Contrato,int64
NumSeq,int64
Est Contrato,object
Tarifa,object
CUI,object
CPE,object
Origem,object
NOA,object
NIF,object


#Representatividad de variables comerciales
Calculamos los porcentajes de consistencia y cambio de variables comerciales a nivel de contrato agrupado por `Num Cliente` utilizando los nombres reales detectados en el dataset de origen `df`.

In [4]:
import pandas as pd

# Buscamos de manera dinámica la columna que representa 'Escalão Consumo' debido al error de codificación
col_escalao = None
for col in df.columns:
    if 'Escal' in col and 'Consumo' in col:
        col_escalao = col
        break
if col_escalao is None:
    col_escalao = 'EscalÃ£o Consumo' # fallback por defecto

# Mapeo de variables solicitadas a los nombres reales en el DataFrame df
variables_mapeo = {
    'Prod': 'Prod',
    'Tarifa': 'Tarifa',
    'Modo Envio Fact': 'Modo Envio Fact',
    'TP Conta': 'TP Conta',
    'Ciclo Hr': 'Ciclo Hr',
    'Tipo de Comunicação': 'Tipo de ComunicaÃ§Ã£o',
    'Origem': 'Origem',
    'Canal Entrada': 'Canal Entrada',
    'Sub Canal Entrada': 'Sub Canal Entrada',
    'Distrito': 'Distrito',
    'Concelho': 'Concelho',
    'Freguesia': 'Freguesia',
    'Escalão Consumo': col_escalao,
    'Potencia': 'Potencia'
}

resultados_variabilidad = []
total_clientes = df['Num Cliente'].nunique()

for var_presentacion, var_real in variables_mapeo.items():
    if var_real in df.columns:
        # Obtenemos la cantidad de valores únicos de cada variable por cliente (incluyendo nulos/guiones)
        valores_por_cliente = df.groupby('Num Cliente')[var_real].nunique(dropna=False)

        # Cálculo de métricas requeridas
        pct_constante = (valores_por_cliente == 1).mean() * 100
        pct_diferentes = (valores_por_cliente > 1).mean() * 100
        max_valores_distintos = valores_por_cliente.max()

        resultados_variabilidad.append({
            'Variable': var_presentacion,
            'Porcentaje de clientes cuyos contratos presentan un único valor (%)': round(pct_constante, 2),
            'Porcentaje de clientes cuyos contratos presentan valores diferentes (%)': round(pct_diferentes, 2),
            'Máximo número de valores distintos para un mismo cliente': int(max_valores_distintos)
        })
    else:
        print(f"Advertencia: La columna '{var_real}' (mapeada de '{var_presentacion}') no existe en el DataFrame.")

# Crear DataFrame final ordenado
tabla_variabilidad_final = pd.DataFrame(resultados_variabilidad)

display(tabla_variabilidad_final)

,Variable,Porcentaje de clientes cuyos contratos presentan un único valor (%),Porcentaje de clientes cuyos contratos presentan valores diferentes (%),Máximo número de valores distintos para un mismo cliente
0,Prod,77.58,22.42,2
1,Tarifa,74.65,25.35,16
2,Modo Envio Fact,99.29,0.71,2
3,TP Conta,98.27,1.73,3
4,Ciclo Hr,77.26,22.74,4
5,Tipo de Comunicação,77.47,22.53,4
6,Origem,95.62,4.38,7
7,Canal Entrada,98.06,1.94,4
8,Sub Canal Entrada,95.94,4.06,7
9,Distrito,98.78,1.22,9


### 2. Análisis de Calidad de Datos (Nulos, Duplicados y Cardinalidad)
Evaluamos el estado de salud de la información sin realizar alteraciones: analizamos la presencia de valores ausentes por columna, la existencia de duplicados exactos en todo el conjunto de datos y la cantidad de valores únicos.

In [ ]:
# 1. Número y porcentaje de valores ausentes por columna
missing_count = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Valores Ausentes': missing_count,
    'Porcentaje (%)': missing_percent
})
print("Análisis de valores ausentes por columna:")
display(missing_df)

# 2. Número de filas duplicadas exactas
duplicados = df.duplicated().sum()
print(f"\nNúmero de filas duplicadas exactas: {duplicados}\n")

# 3. Número de valores únicos por columna (cardinalidad)
valores_unicos = df.nunique()
unicos_df = pd.DataFrame({
    'Valores Únicos (Cardinalidad)': valores_unicos
})
print("Análisis de valores únicos por columna:")
display(unicos_df)

Análisis de valores ausentes por columna:


,Valores Ausentes,Porcentaje (%)
Tipo Cliente,0,0.000000
Contrato,0,0.000000
NumSeq,0,0.000000
Est Contrato,0,0.000000
Tarifa,0,0.000000
CUI,0,0.000000
CPE,0,0.000000
Origem,0,0.000000
NOA,0,0.000000
NIF,0,0.000000



Número de filas duplicadas exactas: 0

Análisis de valores únicos por columna:


,Valores Únicos (Cardinalidad)
Tipo Cliente,7
Contrato,577213
NumSeq,25
Est Contrato,3
Tarifa,4271
CUI,498863
CPE,498863
Origem,179
NOA,2
NIF,498291


### 3. Descripción Estadística de las Variables
Analizamos la distribución estadística de las variables cuantitativas (media, desviación típica, mínimos, percentiles y máximos) y una visión general de la frecuencia de las variables categóricas u objetos presentes.

In [ ]:
# Resumen descriptivo para variables numéricas
print("Resumen descriptivo de las variables numéricas:")
display(df.describe(include=[np.number]))

# Resumen descriptivo para variables de tipo objeto / categóricas (si existen)
print("\nVisión general de las variables no numéricas / categóricas:")
display(df.describe(exclude=[np.number]))

Resumen descriptivo de las variables numéricas:


,Contrato,NumSeq,Num Cliente,CAE
count,6.643710e+05,664371.000000,6.643710e+05,664371.000000
mean,5.688948e+09,101.410372,1.043711e+07,97024.617598
std,1.759515e+10,2.358386,2.689195e+07,11051.869714
min,1.101000e+03,1.000000,1.000000e+00,1111.000000
25%,6.117646e+08,101.000000,4.975070e+05,99921.000000
50%,1.069613e+09,101.000000,8.610880e+05,99921.000000
75%,1.441059e+09,102.000000,1.152353e+06,99921.000000
max,9.999770e+10,131.000000,9.837066e+07,99931.000000



Visión general de las variables no numéricas / categóricas:


,Tipo Cliente,Est Contrato,Tarifa,CUI,CPE,Origem,NOA,NIF,Modo Envio Fact,DT ActivaÃ§Ã£o,...,nome,Canal Entrada,Sub Canal Entrada,Nome Cliente,Morada Envio,CodigoPostal,LocalPostal,Tipo de ComunicaÃ§Ã£o,Identificador Parceria,Campanha
count,664371,664371,664371,664371,664371,664371,664371,664371,664371,664371,...,664371,664366,664371,664371,664371,664371,664371,664371,664371,653441
unique,7,3,4271,498863,498863,179,2,498291,3,5305,...,10,32,141,454109,520769,106695,4003,7,56272,51
top,91 - Domesticos,12 - Contrato em Vigor,Digital Luz Simples 2601,a62199a2534b4b5084b7c556d6ce7ea3,0a727c132e844389b4e0e3a5794a96a5,CAT RECECAO CGI,N,1ff04213db7640668f6cfa499c816f8d,Envio ElectrÃ³nico,-,...,-,CANAL UNICO,CAT Emissao,197c0b76555c40458bbd9d98772093af,b83fe6eb2f4c4098849d9236eab0947b,5d11a565047647dca3fe7c1fad1eeac8,LISBOA,Rede Inteligente,-,-
freq,612237,639575,23448,122,122,135060,505699,122,478579,2184,...,653150,458409,167902,658,268,895,54910,501998,567612,560210
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### FASE 2: DIAGNÓSTICO DETALLADO DE VARIABLES Y CALIDAD DE DATOS

En esta sección realizaremos un diagnóstico en profundidad sin alterar la estructura ni el contenido del DataFrame original (`df`).

#### 1. Verificación de Nombres de Columnas y Errores de Codificación
Mostramos los nombres exactos y verificamos aquellos que presenten caracteres extraños debido a la codificación (UTF-8, ISO-8859-1, etc.).

In [ ]:
# Listar todas las columnas del DataFrame original
columnas_originales = list(df.columns)
print("Nombres exactos de las columnas detectadas:")
for col in columnas_originales:
    print(f" - '{col}'")

Nombres exactos de las columnas detectadas:
 - 'Tipo Cliente'
 - 'Contrato'
 - 'NumSeq'
 - 'Est Contrato'
 - 'Tarifa'
 - 'CUI'
 - 'CPE'
 - 'Origem'
 - 'NOA'
 - 'NIF'
 - 'Num Cliente'
 - 'Modo Envio Fact'
 - 'DT ActivaÃ§Ã£o'
 - 'DT Mod'
 - 'DT Bax'
 - 'Data CriaÃ§Ã£o'
 - 'Prod'
 - 'TP Conta'
 - 'Ciclo Hr'
 - 'Distrito'
 - 'Concelho'
 - 'Freguesia'
 - 'Email'
 - 'CAE'
 - 'Potencia'
 - 'EscalÃ£o Consumo'
 - 'nome'
 - 'Canal Entrada'
 - 'Sub Canal Entrada'
 - 'Nome Cliente'
 - 'Morada Envio'
 - 'CodigoPostal'
 - 'LocalPostal'
 - 'Tipo de ComunicaÃ§Ã£o'
 - 'Identificador Parceria'
 - 'Campanha'


#### 2. Perfilado de Variables Clave
Analizamos el tipo de dato, nulos nativos detectados por pandas, y los 5 valores distintos más frecuentes de las columnas clave relacionadas con el cliente, contrato, tarifas, potencia, escalones de consumo y fechas.

In [ ]:
# Selección de columnas clave para el perfilado
columnas_clave = [
    'Num Cliente', 'Contrato', 'Tipo Cliente', 'Est Contrato',
    'Tarifa', 'Prod', 'Potencia', 'EscalÃ£o Consumo',
    'Data CriaÃ§Ã£o', 'DT ActivaÃ§Ã£o', 'DT Mod', 'DT Bax'
]

print("=== Perfilado de Variables Clave ===\n")
for col in columnas_clave:
    if col in df.columns:
        print(f"Columna: '{col}'")
        print(f" - Tipo de dato: {df[col].dtype}")
        print(f" - Nulos reales (NaN): {df[col].isnull().sum()} ({df[col].isnull().mean()*100:.4f}%)")
        print(" - Top 5 valores más frecuentes (con conteos):")
        print(df[col].value_counts(dropna=False).head(5))
        print("-"*50)
    else:
        print(f"Columna '{col}' no encontrada en el DataFrame.\n")

=== Perfilado de Variables Clave ===

Columna: 'Num Cliente'
 - Tipo de dato: int64
 - Nulos reales (NaN): 0 (0.0000%)
 - Top 5 valores más frecuentes (con conteos):
Num Cliente
994258     122
651710      94
761290      62
1162033     59
1220149     54
Name: count, dtype: int64
--------------------------------------------------
Columna: 'Contrato'
 - Tipo de dato: int64
 - Nulos reales (NaN): 0 (0.0000%)
 - Top 5 valores más frecuentes (con conteos):
Contrato
547152101     4
617990102     2
1031147101    2
1265589101    2
1264658101    2
Name: count, dtype: int64
--------------------------------------------------
Columna: 'Tipo Cliente'
 - Tipo de dato: object
 - Nulos reales (NaN): 0 (0.0000%)
 - Top 5 valores más frecuentes (con conteos):
Tipo Cliente
91 - Domesticos                 612237
92 - NÃ£o DomÃ©sticos            52116
30 - Autarquias                      8
71 - AdministraÃ§Ã£o Central         6
-                                    2
Name: count, dtype: int64
---------------

#### 3. Identificación y Frecuencia del Valor Literal `"-"`
Analizamos de manera exhaustiva en qué columnas aparece el string `"-"` actuando como un nulo enmascarado, clasificándolas según su función de negocio.

In [ ]:
# Identificar la presencia del literal "-" en todo el dataframe
frecuencia_guion = {}
total_filas = len(df)

for col in df.columns:
    conteo = (df[col] == "-").sum()
    if conteo > 0:
        frecuencia_guion[col] = {
            'conteo': conteo,
            'porcentaje': (conteo / total_filas) * 100
        }

# Presentar el reporte clasificado por naturaleza potencial de la variable
guiones_df = pd.DataFrame(frecuencia_guion).T
print("=== Presencia del carácter literal '-' por columna ===\n")
display(guiones_df)

=== Presencia del carácter literal '-' por columna ===



,conteo,porcentaje
Tipo Cliente,2.0,0.000301
Origem,33.0,0.004967
Modo Envio Fact,2.0,0.000301
DT ActivaÃ§Ã£o,2184.0,0.328732
DT Mod,371546.0,55.924476
DT Bax,664371.0,100.000000
TP Conta,2.0,0.000301
Ciclo Hr,134532.0,20.249529
Distrito,8.0,0.001204
Concelho,8.0,0.001204


#### 4. Examen de Formatos y Valores en `Potencia` y `Escalão Consumo`
Analizamos detalladamente las categorías, estructuras y formatos contenidos en estas dos variables clave para la posterior segmentación.

In [ ]:
print("=== Distribución de formatos en 'Potencia' ===")
display(df['Potencia'].value_counts(dropna=False))

print("\n=== Distribución de formatos en 'Escalão Consumo' ===")
escalao_col = 'EscalÃ£o Consumo' if 'EscalÃ£o Consumo' in df.columns else 'Escalão Consumo'
display(df[escalao_col].value_counts(dropna=False))

=== Distribución de formatos en 'Potencia' ===


,count
Potencia,
EDE130,197026
EDE140,147699
-,134532
EDE133,78457
EDE137,41832
EDE150,30317
EDE160,11426
EDE180,8842
EDE110,7144



=== Distribución de formatos en 'Escalão Consumo' ===


,count
EscalÃ£o Consumo,
-,529839
BP < (0 - 220),95779
BP < (221 - 500),33098
BP < (500 - 1000),4560
BP < (1001 - 10000),1095


#### 5. Distribución de Contratos por Cliente y Cardinalidad de Identificadores
Evaluamos la unicidad de las variables identificadoras y la distribución real del volumen de contratos que posee cada cliente.

In [ ]:
# Cardinalidades de los campos identificadores
identificadores = ['Num Cliente', 'Contrato', 'NIF', 'CUI', 'CPE']
print("=== Cardinalidad de Identificadores ===\n")
for id_col in identificadores:
    if id_col in df.columns:
        print(f" - '{id_col}': {df[id_col].nunique()} valores únicos sobre {len(df)} filas.")

# Cálculo de la distribución de contratos por cliente
print("\n=== Distribución de número de contratos por cliente ===\n")
contratos_por_cliente = df.groupby('Num Cliente')['Contrato'].nunique()
resumen_contratos = contratos_por_cliente.value_counts().sort_index()

print("Frecuencia del número de contratos por cliente:")
for num_contratos, cant_clientes in resumen_contratos.items():
    porcentaje = (cant_clientes / len(contratos_por_cliente)) * 100
    print(f" - Clientes con {num_contratos} contrato(s): {cant_clientes} ({porcentaje:.4f}% del total de clientes)")

print("\nEstadísticas descriptivas de contratos por cliente:")
display(contratos_por_cliente.describe())

=== Cardinalidad de Identificadores ===

 - 'Num Cliente': 498863 valores únicos sobre 664371 filas.
 - 'Contrato': 577213 valores únicos sobre 664371 filas.
 - 'NIF': 498291 valores únicos sobre 664371 filas.
 - 'CUI': 498863 valores únicos sobre 664371 filas.
 - 'CPE': 498863 valores únicos sobre 664371 filas.

=== Distribución de número de contratos por cliente ===

Frecuencia del número de contratos por cliente:
 - Clientes con 1 contrato(s): 437455 (87.6904% del total de clientes)
 - Clientes con 2 contrato(s): 51693 (10.3622% del total de clientes)
 - Clientes con 3 contrato(s): 6614 (1.3258% del total de clientes)
 - Clientes con 4 contrato(s): 1720 (0.3448% del total de clientes)
 - Clientes con 5 contrato(s): 604 (0.1211% del total de clientes)
 - Clientes con 6 contrato(s): 285 (0.0571% del total de clientes)
 - Clientes con 7 contrato(s): 166 (0.0333% del total de clientes)
 - Clientes con 8 contrato(s): 96 (0.0192% del total de clientes)
 - Clientes con 9 contrato(s): 72 (0

,Contrato
count,498863.000000
mean,1.157638
std,0.603051
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,121.000000


#### 6. Examen de `Tipo Cliente`
Análisis de las frecuencias de las categorías originales para identificar exactamente todas las representaciones de los clientes domésticos.

In [ ]:
print("=== Categorías y frecuencias en 'Tipo Cliente' ===")
display(df['Tipo Cliente'].value_counts(dropna=False))

=== Categorías y frecuencias en 'Tipo Cliente' ===


,count
Tipo Cliente,
91 - Domesticos,612237
92 - NÃ£o DomÃ©sticos,52116
30 - Autarquias,8
71 - AdministraÃ§Ã£o Central,6
-,2
72 - AdministraÃ§Ã£o Regional,1
61 - Sector publico,1


#### 7. Determinación del Tipo de Suministro / Productos
Analizamos qué columnas y combinaciones de valores nos indican si un contrato o cliente tiene contratado el servicio de electricidad, de gas o de ambos.

In [ ]:
print("=== Distribución de la variable 'Prod' ===")
display(df['Prod'].value_counts(dropna=False))

# Inspeccionamos la relación de suministro a través del CPE (Electricidad) y CUI (Gas) y su presencia
print("\n=== Muestra de relación entre CPE, CUI y Prod (Primeros 10 registros) ===")
display(df[['Prod', 'CPE', 'CUI']].head(10))

=== Distribución de la variable 'Prod' ===


,count
Prod,
EB - Elect. Baixa TensÃ£o,529839
GP - GÃ¡s,134532



=== Muestra de relación entre CPE, CUI y Prod (Primeros 10 registros) ===


,Prod,CPE,CUI
0,GP - GÃ¡s,d9d117f973fc45b8be71298fefbcb138,1c170b569932410bb0ed9d19273d298e
1,GP - GÃ¡s,ccb05374d2124e84bc22e1e061b1a6f3,869651e2927649638b07f4fc7b9c13b8
2,GP - GÃ¡s,bbdc2f80966e46489491aa0915b46f33,61d64e5682b644f7ad1f4ef5e629e92e
3,GP - GÃ¡s,d0855b4779b040268d81fd89bc45d174,ec8b1ea8d3a44fed9bfc705046dd7fbb
4,GP - GÃ¡s,3077a60e9266487688f78e5ffe01f362,54b35d02494b43c291c91bb5972050bf
5,GP - GÃ¡s,405609af7c454f11a787a323a4415f8e,e773fae587d34003baa543f19c64da77
6,GP - GÃ¡s,1ad0496f017e488ba8f0c67a214ecd90,d06c768c0d38461583b779bfb3b15723
7,GP - GÃ¡s,5d0309c303924986a0f47ebaeb2eec41,b9f61b05fae1400fba9be24254bdff4f
8,GP - GÃ¡s,af4270fd657b41518438c314490f8a22,5c817d1a39fb480d851e8b1b3e0ce1c0
9,GP - GÃ¡s,8df013d9f96544bd9911e0b07a311255,e253eb43b2944bc68dd12d01a1270d45


### FASE 3: ANÁLISIS DE RELACIONES MULTIDIMENSIONALES (CLIENTE - CONTRATO - PRODUCTO)

En esta sección analizaremos cómo se estructuran las carteras de contratos y productos de cada cliente y la viabilidad de su posterior consolidación en un modelo de una sola fila por cliente.

#### 1. Auditoría de Codificación y Valores Representativos
Inspeccionamos los nombres exactos almacenados internamente por Pandas y mostramos algunos valores originales de aquellas columnas que presentan caracteres especiales.

In [ ]:
# Columnas con sospecha de problemas de codificación
columnas_conflicto = [
    'DT ActivaÃ§Ã£o', 'Data CriaÃ§Ã£o', 'EscalÃ£o Consumo',
    'Tipo de ComunicaÃ§Ã£o', 'Est Contrato', 'Tarifa', 'Prod'
]

print("=== Muestra de valores originales y su codificación ===\n")
for col in columnas_conflicto:
    if col in df.columns:
        print(f"Columna en Pandas: '{col}'")
        print("Primeros 3 valores representativos:")
        print(df[col].dropna().unique()[:3])
        print("-"*50)

=== Muestra de valores originales y su codificación ===

Columna en Pandas: 'DT ActivaÃ§Ã£o'
Primeros 3 valores representativos:
[datetime.datetime(2026, 6, 24, 0, 0) datetime.datetime(2026, 6, 23, 0, 0)
 datetime.datetime(2026, 6, 22, 0, 0)]
--------------------------------------------------
Columna en Pandas: 'Data CriaÃ§Ã£o'
Primeros 3 valores representativos:
<DatetimeArray>
['2026-06-23 00:00:00', '2026-06-17 00:00:00', '2026-06-22 00:00:00']
Length: 3, dtype: datetime64[ns]
--------------------------------------------------
Columna en Pandas: 'EscalÃ£o Consumo'
Primeros 3 valores representativos:
['BP < (0 - 220)' 'BP < (221 - 500)' '-']
--------------------------------------------------
Columna en Pandas: 'Tipo de ComunicaÃ§Ã£o'
Primeros 3 valores representativos:
['-' 'Rede Inteligente' 'Rede Inteligente com saldo quarto-horÃ¡rio']
--------------------------------------------------
Columna en Pandas: 'Est Contrato'
Primeros 3 valores representativos:
['14 - Contrato Activado nÃ

#### 2. Análisis Multiproducto y Dualidad Contractual
Calculamos la distribución exacta de productos (electricidad y gas) por cliente, mapeamos cuántos contratos posee cada uno y profundizamos específicamente en el segmento de clientes con exactamente dos contratos.

In [ ]:
# 1. Total de clientes y contratos únicos
clientes_unicos = df['Num Cliente'].nunique()
contratos_unicos = df['Contrato'].nunique()

print(f"Clientes únicos: {clientes_unicos}")
print(f"Contratos únicos: {contratos_unicos}\n")

# 2. Productos distintos por cliente
print("=== Distribución del número de productos por cliente ===")
prod_por_cliente = df.groupby('Num Cliente')['Prod'].nunique()
display(prod_por_cliente.value_counts(normalize=True) * 100)

# 3. Distribución de servicios contratados (Electricidad, Gas o Ambos/Dual)
print("\n=== Distribución de tipos de servicios por cliente ===")
servicios_cliente = df.groupby('Num Cliente')['Prod'].apply(set)

def clasificar_servicios(s_set):
    contiene_luz = any('EB' in s or 'Elect' in s for s in s_set)
    contiene_gas = any('GP' in s or 'GÃ¡s' in s or 'Gás' in s for s in s_set)
    if contiene_luz and contiene_gas:
        return 'Dual (Luz y Gas)'
    elif contiene_luz:
        return 'Solo Luz'
    elif contiene_gas:
        return 'Solo Gas'
    else:
        return 'Otro / Indefinido'

tipos_suministro = servicios_cliente.apply(clasificar_servicios).value_counts()
display(pd.DataFrame({'Clientes': tipos_suministro, 'Porcentaje (%)': (tipos_suministro / clientes_unicos) * 100}))

# 4. Análisis profundo de clientes con exactamente 2 contratos
print("\n=== Análisis de clientes con exactamente 2 contratos ===")
clientes_con_dos_contratos = contratos_por_cliente[contratos_por_cliente == 2].index
df_dos_contratos = df[df['Num Cliente'].isin(clientes_con_dos_contratos)]

servicios_dos_contratos = df_dos_contratos.groupby('Num Cliente')['Prod'].apply(list)
combinaciones_dos_contratos = servicios_dos_contratos.apply(lambda x: " + ".join(sorted(x))).value_counts()
display(pd.DataFrame({'Frecuencia': combinaciones_dos_contratos, 'Porcentaje (%)': (combinaciones_dos_contratos / len(clientes_con_dos_contratos)) * 100}))

Clientes únicos: 498863
Contratos únicos: 577213

=== Distribución del número de productos por cliente ===


,proportion
Prod,
1,77.57681
2,22.42319



=== Distribución de tipos de servicios por cliente ===


,Clientes,Porcentaje (%)
Prod,,
Solo Luz,366953,73.557871
Dual (Luz y Gas),111861,22.423190
Solo Gas,20049,4.018939



=== Análisis de clientes con exactamente 2 contratos ===


,Frecuencia,Porcentaje (%)
Prod,,
EB - Elect. Baixa TensÃ£o + GP - GÃ¡s,23948,46.327356
EB - Elect. Baixa TensÃ£o + EB - Elect. Baixa TensÃ£o,21542,41.672954
EB - Elect. Baixa TensÃ£o + EB - Elect. Baixa TensÃ£o + GP - GÃ¡s,5064,9.796297
EB - Elect. Baixa TensÃ£o + EB - Elect. Baixa TensÃ£o + GP - GÃ¡s + GP - GÃ¡s,713,1.379297
EB - Elect. Baixa TensÃ£o + GP - GÃ¡s + GP - GÃ¡s,268,0.518445
GP - GÃ¡s + GP - GÃ¡s,158,0.305651


#### 3. Consistencia de Atributos por Cliente (Estado de Contrato y Tipo de Cliente)
Analizamos si existen clientes cuyos contratos se encuentran en diferentes estados comerciales, o si presentan discrepancias en su catalogación de tipo de cliente.

In [ ]:
# 1. Discrepancias en Estado de Contrato por cliente
estados_por_cliente = df.groupby('Num Cliente')['Est Contrato'].nunique()
clientes_multi_estado = (estados_por_cliente > 1).sum()
print(f"Clientes con contratos en múltiples estados diferentes: {clientes_multi_estado} ({ (clientes_multi_estado / clientes_unicos)*100 :.4f}%)\n")

# 2. Discrepancias en Tipo de Cliente
tipos_por_cliente = df.groupby('Num Cliente')['Tipo Cliente'].nunique()
clientes_multi_tipo = (tipos_por_cliente > 1).sum()
print(f"Clientes con múltiples tipos de cliente asignados: {clientes_multi_tipo} ({ (clientes_multi_tipo / clientes_unicos)*100 :.4f}%)\n")

if clientes_multi_tipo > 0:
    print("Muestra de discrepancias en Tipo de Cliente:")
    ejemplo_discrepancia = tipos_por_cliente[tipos_por_cliente > 1].index[:3]
    display(df[df['Num Cliente'].isin(ejemplo_discrepancia)][['Num Cliente', 'Contrato', 'Tipo Cliente']])

Clientes con contratos en múltiples estados diferentes: 2311 (0.4633%)

Clientes con múltiples tipos de cliente asignados: 963 (0.1930%)

Muestra de discrepancias en Tipo de Cliente:


,Num Cliente,Contrato,Tipo Cliente
638602,670,682101,91 - Domesticos
638789,904,915101,91 - Domesticos
638970,282,282101,91 - Domesticos
650826,904,915101,92 - NÃ£o DomÃ©sticos
651971,282,282101,92 - NÃ£o DomÃ©sticos
658954,670,682101,92 - NÃ£o DomÃ©sticos


#### 4. Integridad de los Identificadores del Contrato
Comprobamos si hay algún contrato que aparezca asociado a más de un cliente, lo que podría indicar una inconsistencia grave de bases de datos relacionales.

In [ ]:
clientes_por_contrato = df.groupby('Contrato')['Num Cliente'].nunique()
contratos_multi_cliente = (clientes_por_contrato > 1).sum()

print(f"Número de contratos asociados a más de un cliente diferente: {contratos_multi_cliente}")
if contratos_multi_cliente > 0:
    print("Ejemplos de contratos compartidos:")
    ejemplo_contratos = clientes_por_contrato[clientes_por_contrato > 1].index[:3]
    display(df[df['Contrato'].isin(ejemplo_contratos)][['Contrato', 'Num Cliente', 'Tipo Cliente']])

Número de contratos asociados a más de un cliente diferente: 290
Ejemplos de contratos compartidos:


,Contrato,Num Cliente,Tipo Cliente
325517,4493101,80381650,91 - Domesticos
340134,144101,80004757,91 - Domesticos
637023,4493101,4551,91 - Domesticos
637544,2703101,2710,91 - Domesticos
637614,2703101,83905319,91 - Domesticos
638999,144101,145,91 - Domesticos


### FASE 4: INVESTIGACIONES DIAGNÓSTICAS DE CALIDAD Y CONSOLIDACIÓN DE CLIENTES

En esta sección implementamos los códigos de diagnóstico requeridos para extraer la evidencia cuantitativa de cara a definir las reglas de negocio de la consolidación, respetando la directriz de no modificar el DataFrame original `df`.

#### 1. Investigación de Contratos Asociados a Múltiples Clientes

Analizamos en profundidad los contratos duplicados entre diferentes números de cliente para evaluar si las variables temporales o de estado permiten resolver la asignación de forma fiable.

In [ ]:
# Identificamos los contratos asociados a más de un cliente
clientes_por_contrato = df.groupby('Contrato')['Num Cliente'].nunique()
contratos_conflictivos = clientes_por_contrato[clientes_por_contrato > 1].index

df_conflictos_contrato = df[df['Contrato'].isin(contratos_conflictivos)].copy()

print(f"Número de contratos en conflicto: {len(contratos_conflictivos)}")
print(f"Número de registros implicados: {len(df_conflictos_contrato)}")
print(f"Número de clientes únicos implicados: {df_conflictos_contrato['Num Cliente'].nunique()}")

# Mostramos una muestra representativa de 3 contratos en conflicto con su historial temporal y comercial
columnas_analisis_contrato = [
    'Contrato', 'Num Cliente', 'Data CriaÃ§Ã£o', 'DT ActivaÃ§Ã£o', 'DT Mod', 'Est Contrato', 'Prod'
]

print(
    "\n=== Muestra Detallada de Contratos en Conflicto (Primeros 3 Casos con todos sus registros) ==="
)
ejemplos_contratos = contratos_conflictivos[:3]
display(
    df_conflictos_contrato[
        df_conflictos_contrato['Contrato'].isin(ejemplos_contratos)
    ][columnas_analisis_contrato].sort_values(by=['Contrato', 'Data CriaÃ§Ã£o'])
)

# Evaluamos el estado contractual de estos registros para ver si alguno está activo/inactivo de forma concluyente
print("\n=== Distribución de 'Est Contrato' en registros de contratos con conflicto ===")
display(df_conflictos_contrato['Est Contrato'].value_counts(dropna=False))

Número de contratos en conflicto: 290
Número de registros implicados: 580
Número de clientes únicos implicados: 578

=== Muestra Detallada de Contratos en Conflicto (Primeros 3 Casos con todos sus registros) ===


,Contrato,Num Cliente,Data CriaÃ§Ã£o,DT ActivaÃ§Ã£o,DT Mod,Est Contrato,Prod
638999,144101,145,2015-07-20,2015-08-12 00:00:00,2025-12-23 00:00:00,12 - Contrato em Vigor,GP - GÃ¡s
340134,144101,80004757,2024-04-17,2024-04-15 00:00:00,2025-12-23 00:00:00,12 - Contrato em Vigor,EB - Elect. Baixa TensÃ£o
637544,2703101,2710,2015-10-26,2015-10-30 00:00:00,2021-09-01 00:00:00,12 - Contrato em Vigor,GP - GÃ¡s
637614,2703101,83905319,2016-05-16,2015-10-23 00:00:00,-,12 - Contrato em Vigor,EB - Elect. Baixa TensÃ£o
637023,4493101,4551,2015-11-30,2015-12-11 00:00:00,2024-06-21 00:00:00,12 - Contrato em Vigor,GP - GÃ¡s
325517,4493101,80381650,2024-06-21,2024-06-19 00:00:00,2024-06-21 00:00:00,12 - Contrato em Vigor,EB - Elect. Baixa TensÃ£o



=== Distribución de 'Est Contrato' en registros de contratos con conflicto ===


,count
Est Contrato,
12 - Contrato em Vigor,557
14 - Contrato Activado nÃ£o Factur.,19
15 - Contrato Pendente de ActivaÃ§Ã£o,4


#### 2. Investigación de Discrepancias en `Tipo Cliente`

Analizamos a los clientes que tienen asignados diferentes tipos de cliente en sus diversos contratos para proponer reglas de resolución (predominancia, estados, etc.).

In [ ]:
# Identificamos clientes con discrepancia en Tipo de Cliente
tipos_por_cliente = df.groupby('Num Cliente')['Tipo Cliente'].nunique()
clientes_discrepantes = tipos_por_cliente[tipos_por_cliente > 1].index

df_discrepantes_tipo = df[df['Num Cliente'].isin(clientes_discrepantes)].copy()

print(f"Clientes únicos con discrepancia de tipo: {len(clientes_discrepantes)}")
print(f"Registros totales asociados a estos clientes: {len(df_discrepantes_tipo)}\n")

# Cruzamos el Tipo de Cliente con el Estado del Contrato para estos clientes conflictivos
print("=== Matriz de Tipo de Cliente vs Estado de Contrato para clientes con discrepancia ===")
matriz_discrepancia = pd.crosstab(
    df_discrepantes_tipo['Tipo Cliente'],
    df_discrepantes_tipo['Est Contrato'],
    margins=True
)
display(matriz_discrepancia)

# Mostramos un ejemplo de clientes con registros contradictorios para ver si las fechas ayudan
print("\n=== Ejemplo de Clientes con Discrepancia en Tipo Cliente ===")
ejemplos_clientes = clientes_discrepantes[:3]
display(
    df_discrepantes_tipo[
        df_discrepantes_tipo['Num Cliente'].isin(ejemplos_clientes)
    ][[
        'Num Cliente', 'Contrato', 'Tipo Cliente', 'Data CriaÃ§Ã£o',
        'DT ActivaÃ§Ã£o', 'Est Contrato', 'Prod'
    ]].sort_values(by=['Num Cliente', 'Data CriaÃ§Ã£o'])
)

Clientes únicos con discrepancia de tipo: 963
Registros totales asociados a estos clientes: 2874

=== Matriz de Tipo de Cliente vs Estado de Contrato para clientes con discrepancia ===


Est Contrato,12 - Contrato em Vigor,14 - Contrato Activado nÃ£o Factur.,15 - Contrato Pendente de ActivaÃ§Ã£o,All
Tipo Cliente,,,,
-,2,0,0,2
30 - Autarquias,3,0,0,3
61 - Sector publico,1,0,0,1
71 - AdministraÃ§Ã£o Central,3,1,0,4
91 - Domesticos,1254,41,3,1298
92 - NÃ£o DomÃ©sticos,1507,58,1,1566
All,2770,100,4,2874



=== Ejemplo de Clientes con Discrepancia en Tipo Cliente ===


,Num Cliente,Contrato,Tipo Cliente,Data CriaÃ§Ã£o,DT ActivaÃ§Ã£o,Est Contrato,Prod
638970,282,282101,91 - Domesticos,2015-07-28,2015-08-13 00:00:00,12 - Contrato em Vigor,GP - GÃ¡s
651971,282,282101,92 - NÃ£o DomÃ©sticos,2016-05-16,2013-07-03 00:00:00,12 - Contrato em Vigor,EB - Elect. Baixa TensÃ£o
638602,670,682101,91 - Domesticos,2015-08-06,2015-08-29 00:00:00,12 - Contrato em Vigor,GP - GÃ¡s
658954,670,682101,92 - NÃ£o DomÃ©sticos,2016-05-16,2011-10-19 00:00:00,12 - Contrato em Vigor,EB - Elect. Baixa TensÃ£o
638789,904,915101,91 - Domesticos,2015-08-14,2015-08-21 00:00:00,12 - Contrato em Vigor,GP - GÃ¡s
650826,904,915101,92 - NÃ£o DomÃ©sticos,2016-05-16,2013-07-13 00:00:00,12 - Contrato em Vigor,EB - Elect. Baixa TensÃ£o


#### 3. Análisis de los Valores `"-"` y Nulos Reales

Estudiamos detalladamente la presencia de nulos nativos y la codificación de ausencias con el carácter `"-"` en la lista de variables críticas de negocio.

In [ ]:
variables_analizar = [
    'Potencia', 'EscalÃ£o Consumo', 'Ciclo Hr', 'DT Bax', 'DT Mod',
    'DT ActivaÃ§Ã£o', 'Identificador Parceria', 'Campanha', 'Tipo de ComunicaÃ§Ã£o'
]

analisis_nulos_guion = []

for col in variables_analizar:
    if col in df.columns:
        nulos_reales = df[col].isnull().sum()
        porc_nulos = (nulos_reales / len(df)) * 100

        conteo_guion = (df[col] == "-").sum()
        porc_guion = (conteo_guion / len(df)) * 100

        # Obtener valores distintos no nulos ni guión (Top 3)
        valores_reales = df[(df[col].notnull()) & (df[col] != "-")][col]
        top_valores = list(valores_reales.value_counts().head(3).index)

        analisis_nulos_guion.append({
            'Variable': col,
            'Nulos Reales (NaN)': f"{nulos_reales} ({porc_nulos:.3f}%)",
            'Guiones "-"': f"{conteo_guion} ({porc_guion:.3f}%)",
            'Top 3 Valores Válidos': top_valores
        })

df_analisis_nulos = pd.DataFrame(analisis_nulos_guion)
display(df_analisis_nulos)

# Análisis cruzado de Potencia y Escalão de Consumo con el Producto
print("\n=== Relación de 'Potencia' con Tipo de Producto ('Prod') ===")
display(pd.crosstab(df['Potencia'] == "-", df['Prod'], margins=True))

print("\n=== Relación de 'Escalão Consumo' con Tipo de Producto ('Prod') ===")
escalao_real_col = 'EscalÃ£o Consumo' if 'EscalÃ£o Consumo' in df.columns else 'Escalão Consumo'
display(pd.crosstab(df[escalao_real_col] == "-", df['Prod'], margins=True))

/tmp/ipykernel_2836/2071403977.py:18: FutureWarning: The behavior of value_counts with object-dtype is deprecated. In a future version, this will *not* perform dtype inference on the resulting index. To retain the old behavior, use `result.index = result.index.infer_objects()`
  top_valores = list(valores_reales.value_counts().head(3).index)
/tmp/ipykernel_2836/2071403977.py:18: FutureWarning: The behavior of value_counts with object-dtype is deprecated. In a future version, this will *not* perform dtype inference on the resulting index. To retain the old behavior, use `result.index = result.index.infer_objects()`
  top_valores = list(valores_reales.value_counts().head(3).index)


,Variable,Nulos Reales (NaN),"Guiones ""-""",Top 3 Valores Válidos
0,Potencia,0 (0.000%),134532 (20.250%),"[EDE130, EDE140, EDE133]"
1,EscalÃ£o Consumo,0 (0.000%),529839 (79.750%),"[BP < (0 - 220), BP < (221 - 500), BP < (500 -..."
2,Ciclo Hr,0 (0.000%),134532 (20.250%),"[SC, D, SF]"
3,DT Bax,0 (0.000%),664371 (100.000%),[]
4,DT Mod,0 (0.000%),371546 (55.924%),"[2022-07-15 00:00:00, 2021-09-01 00:00:00, 202..."
5,DT ActivaÃ§Ã£o,0 (0.000%),2184 (0.329%),"[2026-04-01 00:00:00, 2026-03-13 00:00:00, 202..."
6,Identificador Parceria,0 (0.000%),567612 (85.436%),"[0, 1, 18]"
7,Campanha,10930 (1.645%),560210 (84.322%),"[MAIS ENERGIA, RECUPERACAO, QUEROMAIS 3]"
8,Tipo de ComunicaÃ§Ã£o,0 (0.000%),142029 (21.378%),"[Rede Inteligente, Rede Inteligente com saldo ..."



=== Relación de 'Potencia' con Tipo de Producto ('Prod') ===


Prod,EB - Elect. Baixa TensÃ£o,GP - GÃ¡s,All
Potencia,,,
False,529839,0,529839
True,0,134532,134532
All,529839,134532,664371



=== Relación de 'Escalão Consumo' con Tipo de Producto ('Prod') ===


Prod,EB - Elect. Baixa TensÃ£o,GP - GÃ¡s,All
EscalÃ£o Consumo,,,
False,0,134532,134532
True,529839,0,529839
All,529839,134532,664371


#### 4. Evaluación de las Variables Candidatas a la Agregación

Analizaremos la cardinalidad y distribución de variables críticas que se pretenden resumir a nivel de cliente.

In [ ]:
variables_agregacion = [
    'Prod', 'Tarifa', 'Potencia', 'EscalÃ£o Consumo', 'Est Contrato', 'Tipo Cliente', 'Data CriaÃ§Ã£o'
]

for col in variables_agregacion:
    if col in df.columns:
        print(f"Variable: '{col}'")
        print(f" - Cardinalidad (valores únicos totales): {df[col].nunique()}")
        print(f" - Distribución de frecuencias principales (Top 5):")
        print(df[col].value_counts(dropna=False).head(5))
        print("-" * 60)

Variable: 'Prod'
 - Cardinalidad (valores únicos totales): 2
 - Distribución de frecuencias principales (Top 5):
Prod
EB - Elect. Baixa TensÃ£o    529839
GP - GÃ¡s                    134532
Name: count, dtype: int64
------------------------------------------------------------
Variable: 'Tarifa'
 - Cardinalidad (valores únicos totales): 4271
 - Distribución de frecuencias principales (Top 5):
Tarifa
Digital Luz Simples 2601    23448
+Energia Luz Simp 2601      20452
+Energia Luz Simp 2604      13138
Quero+ Luz Simples 2406     10286
Aniv. Luz Simples 2410       9542
Name: count, dtype: int64
------------------------------------------------------------
Variable: 'Potencia'
 - Cardinalidad (valores únicos totales): 14
 - Distribución de frecuencias principales (Top 5):
Potencia
EDE130    197026
EDE140    147699
-         134532
EDE133     78457
EDE137     41832
Name: count, dtype: int64
------------------------------------------------------------
Variable: 'EscalÃ£o Consumo'
 - Cardinalid

#### 5. Evaluación de la Variable de Antigüedad

Inspeccionamos los rangos de fechas disponibles en `Data Criação`, `DT Activação` y `DT Mod` para determinar la solidez matemática de la métrica de antigüedad.

In [ ]:
# Convertimos temporalmente variables para validar rangos de fecha de forma segura sin sobreescribir el DF original
fecha_creacion = pd.to_datetime(df['Data CriaÃ§Ã£o'], errors='coerce')

# Filtrando nulos y guiones de activación y modificación para su análisis
fecha_act_aux = df['DT ActivaÃ§Ã£o'].copy()
fecha_act_aux = pd.to_datetime(fecha_act_aux.replace('-', np.nan), errors='coerce')

print("=== Estadísticas de Cobertura y Rangos de Fechas ===")
print(f"'Data Criação' - Registros convertibles a fecha: {fecha_creacion.notnull().sum()} de {len(df)}")
print(f"'Data Criação' - Rango temporal detectado: {fecha_creacion.min()} hasta {fecha_creacion.max()}")
print(f"'DT Activação' - Registros convertibles a fecha: {fecha_act_aux.notnull().sum()} de {len(df)}")
print(f"'DT Activação' - Rango temporal detectado: {fecha_act_aux.min()} hasta {fecha_act_aux.max()}")

# Verificación de fechas anómalas (ej. fechas en el futuro como 2026)
print(f"\nRegistros con creación en el año 2026: {(fecha_creacion.dt.year == 2026).sum()}")
print(f"Registros con creación en el año 2016: {(fecha_creacion.dt.year == 2016).sum()}")

=== Estadísticas de Cobertura y Rangos de Fechas ===
'Data Criação' - Registros convertibles a fecha: 664371 de 664371
'Data Criação' - Rango temporal detectado: 2015-07-10 00:00:00 hasta 2026-06-24 00:00:00
'DT Activação' - Registros convertibles a fecha: 662187 de 664371
'DT Activação' - Rango temporal detectado: 2009-06-17 00:00:00 hasta 2026-06-24 00:00:00

Registros con creación en el año 2026: 133205
Registros con creación en el año 2016: 36226


/tmp/ipykernel_2836/2919576303.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  fecha_act_aux = pd.to_datetime(fecha_act_aux.replace('-', np.nan), errors='coerce')


### FASE 4.1: ANÁLISIS CRÍTICO Y CUANTIFICACIÓN EMPÍRICA DE REGLAS DE CONSOLIDACIÓN

In [ ]:
# 1. Investigación crítica de los 290 contratos asociados a múltiples clientes
contratos_multi = df.groupby('Contrato').filter(lambda x: x['Num Cliente'].nunique() > 1)

print("=== 1. Análisis de Unicidad por Combinación Contrato + Prod ===")
print(f"Total de registros con contratos compartidos: {len(contratos_multi)}")

# Comprobamos si la combinación Contrato + Prod es única
comb_contrato_prod = contratos_multi.groupby(['Contrato', 'Prod']).size()
max_registros_por_combinacion = comb_contrato_prod.max()
print(f"Máximo de registros por combinación 'Contrato' + 'Prod': {max_registros_por_combinacion}")

# Analizamos si CPE y CUI están segregados correctamente en estos contratos
print("\nMuestra de los campos CPE y CUI para contratos compartidos:")
display(contratos_multi[['Contrato', 'Num Cliente', 'Prod', 'CPE', 'CUI', 'Est Contrato', 'Data CriaÃ§Ã£o']].head(6))

# Verificar si hay CPEs o CUIs duplicados entre distintos clientes en estos registros
cpes_compartidos = contratos_multi.groupby('CPE')['Num Cliente'].nunique().max()
cuis_compartidos = contratos_multi.groupby('CUI')['Num Cliente'].nunique().max()
print(f"¿Hay algún CPE asociado a más de un cliente?: {'Sí' if cpes_compartidos > 1 else 'No'}")
print(f"¿Hay algún CUI asociado a más de un cliente?: {'Sí' if cuis_compartidos > 1 else 'No'}")

=== 1. Análisis de Unicidad por Combinación Contrato + Prod ===
Total de registros con contratos compartidos: 580
Máximo de registros por combinación 'Contrato' + 'Prod': 1

Muestra de los campos CPE y CUI para contratos compartidos:


,Contrato,Num Cliente,Prod,CPE,CUI,Est Contrato,Data CriaÃ§Ã£o
148,1395205102,986340,GP - GÃ¡s,fec1dc27737e4c8582222beee3b1e018,c6fb07b4660740fa9497e16121e3301b,14 - Contrato Activado nÃ£o Factur.,2026-06-23
434,1395205102,997548,EB - Elect. Baixa TensÃ£o,20ec68681b1d42b3b45b06e789dc02a4,0c2004b9a0c24fa2bc8d1ab2995dc335,14 - Contrato Activado nÃ£o Factur.,2026-06-23
2206,144035103,872432,EB - Elect. Baixa TensÃ£o,0e01e85c313041d18a9e1622d31d7c59,2c9fa8ca48b64b22a54f7d9b53945d0f,14 - Contrato Activado nÃ£o Factur.,2026-06-24
3069,80037102,1242329,EB - Elect. Baixa TensÃ£o,609219c37cbc4531a4d8b7e4448e23f8,68d7c1fd52d94555959d0450be93b1e3,14 - Contrato Activado nÃ£o Factur.,2026-06-23
3110,1375954102,1027388,EB - Elect. Baixa TensÃ£o,7484f4e004cd4ed996ae1926bdc8b5ef,939a9ad4f09b4c639539648ee9589787,14 - Contrato Activado nÃ£o Factur.,2026-06-23
3210,225737102,1242447,EB - Elect. Baixa TensÃ£o,e7222178db22480aabdaf31da558fa43,7b74706f563d4f1787e024d0657a169d,14 - Contrato Activado nÃ£o Factur.,2026-06-23


¿Hay algún CPE asociado a más de un cliente?: No
¿Hay algún CUI asociado a más de un cliente?: No


In [ ]:
# 2. Simulación cuantitativa de las 4 reglas de resolución para los 963 clientes con Tipo Cliente discrepante
clientes_discrepantes_ids = df.groupby('Num Cliente').filter(lambda x: x['Tipo Cliente'].nunique() > 1)['Num Cliente'].unique()
df_disc = df[df['Num Cliente'].isin(clientes_discrepantes_ids)].copy()

print(f"=== 2. Evaluación Cuantitativa de Reglas para {len(clientes_discrepantes_ids)} Clientes Discrepantes ===\n")

# REGLA A: Valor predominante
def calcular_predominante(group):
    counts = group['Tipo Cliente'].value_counts()
    if len(counts) > 1 and counts.iloc[0] == counts.iloc[1]:
        return "Empate"
    return counts.index[0]

predominantes = df_disc.groupby('Num Cliente').apply(calcular_predominante)
clientes_empate = (predominantes == "Empate").sum()
print(f"Regla A (Valor Predominante): {len(clientes_discrepantes_ids) - clientes_empate} resueltos, {clientes_empate} clientes en empate (sin resolver).")

# REGLA B: Priorizar contrato en vigor ('12 - Contrato em Vigor')
def calcular_en_vigor(group):
    vigores = group[group['Est Contrato'] == '12 - Contrato em Vigor']['Tipo Cliente'].unique()
    if len(vigores) == 1:
        return vigores[0]
    elif len(vigores) > 1:
        return "Conflicto Múltiple en Vigor"
    else:
        return "Ninguno en Vigor"

en_vigor_res = df_disc.groupby('Num Cliente').apply(calcular_en_vigor)
print(f"Regla B (Priorizar En Vigor): {sum(en_vigor_res.isin(['91 - Domesticos', '92 - NÃ£o DomÃ©sticos']))} resueltos de forma única.")
print(en_vigor_res.value_counts())

# REGLA C: Clasificar como No Doméstico si existe al menos un contrato No Doméstico
def calcular_no_domestico_prioritario(group):
    tipos = group['Tipo Cliente'].unique()
    if '92 - NÃ£o DomÃ©sticos' in tipos:
        return '92 - NÃ£o DomÃ©sticos'
    return '91 - Domesticos'

no_dom_res = df_disc.groupby('Num Cliente').apply(calcular_no_domestico_prioritario)
print(f"\nRegla C (Priorizar No Doméstico): Clasificados como No Domésticos: {sum(no_dom_res == '92 - NÃ£o DomÃ©sticos')}, Domésticos: {sum(no_dom_res == '91 - Domesticos')}.")

# REGLA D: Mantener el conflicto identificado sin asignar
print(f"Regla D (Mantener Conflicto): {len(clientes_discrepantes_ids)} clientes etiquetados con bandera de conflicto.")

=== 2. Evaluación Cuantitativa de Reglas para 963 Clientes Discrepantes ===



/tmp/ipykernel_2836/2072043451.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  predominantes = df_disc.groupby('Num Cliente').apply(calcular_predominante)


Regla A (Valor Predominante): 308 resueltos, 655 clientes en empate (sin resolver).
Regla B (Priorizar En Vigor): 13 resueltos de forma única.
Conflicto Múltiple en Vigor    927
Ninguno en Vigor                23
92 - NÃ£o DomÃ©sticos            7
91 - Domesticos                  6
Name: count, dtype: int64

Regla C (Priorizar No Doméstico): Clasificados como No Domésticos: 956, Domésticos: 7.
Regla D (Mantener Conflicto): 963 clientes etiquetados con bandera de conflicto.


/tmp/ipykernel_2836/2072043451.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  en_vigor_res = df_disc.groupby('Num Cliente').apply(calcular_en_vigor)
/tmp/ipykernel_2836/2072043451.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  no_dom_res = df_disc.groupby('Num Cliente').apply(calcular_no_domestico_prioritario)


In [ ]:
# 3. Inspección detallada de Escalão Consumo y Potencia
print("=== 3. Análisis de Escalão Consumo ===")
escalao_col = 'EscalÃ£o Consumo' if 'EscalÃ£o Consumo' in df.columns else 'Escalão Consumo'
print("Categorías de Escalón de Consumo presentes en el dataset:")
display(df[escalao_col].value_counts(dropna=False))

print("\n=== Análisis de códigos de Potencia ===")
print("Códigos únicos de Potencia y sus frecuencias:")
display(df['Potencia'].value_counts(dropna=False))

=== 3. Análisis de Escalão Consumo ===
Categorías de Escalón de Consumo presentes en el dataset:


,count
EscalÃ£o Consumo,
-,529839
BP < (0 - 220),95779
BP < (221 - 500),33098
BP < (500 - 1000),4560
BP < (1001 - 10000),1095



=== Análisis de códigos de Potencia ===
Códigos únicos de Potencia y sus frecuencias:


,count
Potencia,
EDE130,197026
EDE140,147699
-,134532
EDE133,78457
EDE137,41832
EDE150,30317
EDE160,11426
EDE180,8842
EDE110,7144


### FASE 5A.1: LIMPIEZA CONTROLADA Y CONSOLIDACIÓN A NIVEL DE CLIENTE (ALCANCE ACOTADO)

Siguiendo las instrucciones acotadas, crearemos un nuevo DataFrame consolidado `df_clientes_f5` con exactamente una fila por `Num Cliente` basándonos exclusivamente en el DataFrame original `df`, comprobando la unicidad de las claves y detectando conflictos de forma transparente.

In [ ]:
import pandas as pd
import numpy as np

print("=== 1. COMPROBACIÓN DE LOS DATOS DE ORIGEN ===")
num_filas_orig = len(df)
clientes_unicos_orig = df['Num Cliente'].nunique()
contratos_unicos_orig = df['Contrato'].nunique()

print(f"- Número de filas originales: {num_filas_orig}")
print(f"- Clientes únicos originales (Num Cliente): {clientes_unicos_orig}")
print(f"- Contratos únicos originales: {contratos_unicos_orig}")

# Verificar valores observados en Prod y Tipo Cliente
print("\nValores en 'Prod':")
print(df['Prod'].value_counts(dropna=False))
print("\nValores en 'Tipo Cliente':")
print(df['Tipo Cliente'].value_counts(dropna=False))

# Comprobar la unicidad de Contrato y Contrato + Prod
max_contrato_repetido = df.groupby('Contrato').size().max()
max_comb_repetido = df.groupby(['Contrato', 'Prod']).size().max()
print(f"\n- Máximas repeticiones de un mismo 'Contrato': {max_contrato_repetido}")
print(f"- Máximas repeticiones de la combinación 'Contrato' + 'Prod': {max_comb_repetido} (Debe ser 1 para confirmar unicidad absoluta)")

=== 1. COMPROBACIÓN DE LOS DATOS DE ORIGEN ===
- Número de filas originales: 664371
- Clientes únicos originales (Num Cliente): 498863
- Contratos únicos originales: 577213

Valores en 'Prod':
Prod
EB - Elect. Baixa TensÃ£o    529839
GP - GÃ¡s                    134532
Name: count, dtype: int64

Valores en 'Tipo Cliente':
Tipo Cliente
91 - Domesticos                  612237
92 - NÃ£o DomÃ©sticos             52116
30 - Autarquias                       8
71 - AdministraÃ§Ã£o Central          6
-                                     2
72 - AdministraÃ§Ã£o Regional         1
61 - Sector publico                   1
Name: count, dtype: int64

- Máximas repeticiones de un mismo 'Contrato': 4
- Máximas repeticiones de la combinación 'Contrato' + 'Prod': 2 (Debe ser 1 para confirmar unicidad absoluta)


In [ ]:
print("=== 2. CREACIÓN DE INDICADORES DE PRODUCTOS ===")

# Creamos copias de trabajo para evitar warnings de asignación
df_prod = df.copy()
df_prod['Es_Luz'] = df_prod['Prod'].apply(lambda x: 1 if ('EB' in str(x) or 'Elect' in str(x)) else 0)
df_prod['Es_Gas'] = df_prod['Prod'].apply(lambda x: 1 if ('GP' in str(x) or 'GÃ¡s' in str(x) or 'Gás' in str(x)) else 0)

# Agregamos a nivel de Num Cliente
productos_per_cliente = df_prod.groupby('Num Cliente').agg(
    Tiene_Luz=('Es_Luz', 'max'),
    Tiene_Gas=('Es_Gas', 'max')
).reset_index()

productos_per_cliente['Num_Productos'] = productos_per_cliente['Tiene_Luz'] + productos_per_cliente['Tiene_Gas']

def segmentar_cliente(row):
    if row['Tiene_Luz'] == 1 and row['Tiene_Gas'] == 1:
        return 'Dual'
    elif row['Tiene_Luz'] == 1:
        return 'Solo Luz'
    elif row['Tiene_Gas'] == 1:
        return 'Solo Gas'
    else:
        return 'Sin Producto'

productos_per_cliente['Clasificacion_Suministro'] = productos_per_cliente.apply(segmentar_cliente, axis=1)

print("Distribución de tenencia de productos por cliente:")
print(productos_per_cliente['Clasificacion_Suministro'].value_counts())

=== 2. CREACIÓN DE INDICADORES DE PRODUCTOS ===
Distribución de tenencia de productos por cliente:
Clasificacion_Suministro
Solo Luz    366953
Dual        111861
Solo Gas     20049
Name: count, dtype: int64


In [ ]:
print("=== 3. CONSOLIDACIÓN A NIVEL DE CLIENTE ===")

# Contar contratos distintos por cliente
contratos_cliente = df.groupby('Num Cliente')['Contrato'].nunique().reset_index(name='Num_Contratos')

# Identificar conflictos de Tipo Cliente sin modificar o forzar valores
tipos_por_cliente = df.groupby('Num Cliente')['Tipo Cliente'].nunique()
clientes_con_conflicto = tipos_por_cliente[tipos_por_cliente > 1].index

conflictos_df = pd.DataFrame({'Num Cliente': df['Num Cliente'].unique()})
conflictos_df['Conflicto_Tipo_Cliente'] = conflictos_df['Num Cliente'].isin(clientes_con_conflicto).astype(int)

# Unión limpia para crear el DataFrame consolidado
df_clientes_f5 = productos_per_cliente.merge(contratos_cliente, on='Num Cliente', how='left')
df_clientes_f5 = df_clientes_f5.merge(conflictos_df, on='Num Cliente', how='left')

print(f"Consolidación completada. Filas resultantes: {len(df_clientes_f5)}")

=== 3. CONSOLIDACIÓN A NIVEL DE CLIENTE ===
Consolidación completada. Filas resultantes: 498863


In [ ]:
print("=== 4. PANEL DE VALIDACIONES OBLIGATORIAS ===\n")

# 1. Filas y Clientes únicos originales
print(f"- Número de filas originales: {num_filas_orig}")
print(f"- Clientes únicos originales: {clientes_unicos_orig}")

# 2. Filas del nuevo DataFrame y Unicidad de Num Cliente
print(f"- Número de filas de la tabla consolidada: {len(df_clientes_f5)}")
print(f"- ¿Num Cliente es único en la tabla consolidada?: {'Sí' if not df_clientes_f5['Num Cliente'].duplicated().any() else 'No'}")

# 3. Distribución comparativa del Tipo de Suministro
print("\n- Distribución del Tipo de Suministro en la tabla consolidada:")
print(df_clientes_f5['Clasificacion_Suministro'].value_counts())

# 4. Conflictos de Tipo Cliente
cant_conflictos = df_clientes_f5['Conflicto_Tipo_Cliente'].sum()
print(f"\n- Número de clientes con conflictos de 'Tipo Cliente': {cant_conflictos} ({ (cant_conflictos / clientes_unicos_orig)*100 :.4f}% del total)")

# 5. Reconciliación de contratos e identificadores compartidos
clientes_por_contrato = df.groupby('Contrato')['Num Cliente'].nunique()
contratos_compartidos_count = (clientes_por_contrato > 1).sum()
suma_contratos_calculada = df_clientes_f5['Num_Contratos'].sum()

print(f"\n- Número de contratos ('Contrato') compartidos entre múltiples clientes distintos: {contratos_compartidos_count}")
print(f"- Suma de recuentos de contratos calculados por cliente: {suma_contratos_calculada}")
print(f"- Contratos únicos en el dataset original: {contratos_unicos_orig}")
print(f"- Discrepancia matemática en la reconciliación (Suma calculada - Únicos globales): {suma_contratos_calculada - contratos_unicos_orig} (Esperado exactamente 290 debido a los contratos de gas/luz compartidos entre 2 clientes)")

# 6. Resumen de columnas creadas y tipos de datos
print("\n- Resumen de estructura y tipos de datos del nuevo DataFrame:")
display(df_clientes_f5.info())

=== 4. PANEL DE VALIDACIONES OBLIGATORIAS ===

- Número de filas originales: 664371
- Clientes únicos originales: 498863
- Número de filas de la tabla consolidada: 498863
- ¿Num Cliente es único en la tabla consolidada?: Sí

- Distribución del Tipo de Suministro en la tabla consolidada:
Clasificacion_Suministro
Solo Luz    366953
Dual        111861
Solo Gas     20049
Name: count, dtype: int64

- Número de clientes con conflictos de 'Tipo Cliente': 963 (0.1930% del total)

- Número de contratos ('Contrato') compartidos entre múltiples clientes distintos: 290
- Suma de recuentos de contratos calculados por cliente: 577503
- Contratos únicos en el dataset original: 577213
- Discrepancia matemática en la reconciliación (Suma calculada - Únicos globales): 290 (Esperado exactamente 290 debido a los contratos de gas/luz compartidos entre 2 clientes)

- Resumen de estructura y tipos de datos del nuevo DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 498863 entries, 0 to 498862
Data

None

### FASE 5A.1B: VALIDACIÓN FINAL DE LA CONSOLIDACIÓN

Realizamos las comprobaciones empíricas cruzando directamente el DataFrame consolidado `df_clientes_f5` con los datos de origen `df` para garantizar la integridad absoluta de la consolidación sin modificar la base original.

In [ ]:
import pandas as pd
import numpy as np

results_summary = []

print("=== 1. UNICIDAD Y VALORES NULOS EN LA TABLA CONSOLIDADA ===")
total_filas_consolidado = len(df_clientes_f5)
clientes_unicos_orig = df['Num Cliente'].nunique()
clientes_duplicados_cons = df_clientes_f5['Num Cliente'].duplicated().sum()
clientes_nulos_cons = df_clientes_f5['Num Cliente'].isnull().sum()

check_1_passed = (total_filas_consolidado == clientes_unicos_orig) and (clientes_duplicados_cons == 0) and (clientes_nulos_cons == 0)
print(f"- Filas consolidadas: {total_filas_consolidado} (Esperado: {clientes_unicos_orig})")
print(f"- Clientes duplicados: {clientes_duplicados_cons} (Esperado: 0)")
print(f"- Clientes nulos: {clientes_nulos_cons} (Esperado: 0)")
print(f"¿Comprobación 1 superada?: {'SÍ' if check_1_passed else 'NO'}")

results_summary.append({
    'Comprobación': '1. Unicidad y Nulos de Num Cliente',
    'Esperado': f'{clientes_unicos_orig} filas, 0 duplicados, 0 nulos',
    'Observado': f'{total_filas_consolidado} filas, {clientes_duplicados_cons} duplicados, {clientes_nulos_cons} nulos',
    'Estado': 'Superado' if check_1_passed else 'Fallo'
})

=== 1. UNICIDAD Y VALORES NULOS EN LA TABLA CONSOLIDADA ===
- Filas consolidadas: 498863 (Esperado: 498863)
- Clientes duplicados: 0 (Esperado: 0)
- Clientes nulos: 0 (Esperado: 0)
¿Comprobación 1 superada?: SÍ


In [ ]:
print("=== 2. COMPARACIÓN DE LA DISTRIBUCIÓN DE PRODUCTOS (PRE VS POST) ===")

# Recalcular desde el origen original (df)
df_prod_calc = df.copy()
df_prod_calc['Es_Luz'] = df_prod_calc['Prod'].apply(lambda x: 1 if ('EB' in str(x) or 'Elect' in str(x)) else 0)
df_prod_calc['Es_Gas'] = df_prod_calc['Prod'].apply(lambda x: 1 if ('GP' in str(x) or 'GÃ¡s' in str(x) or 'Gás' in str(x)) else 0)

prod_orig = df_prod_calc.groupby('Num Cliente').agg(
    Luz=('Es_Luz', 'max'),
    Gas=('Es_Gas', 'max')
)

def class_suministro(row):
    if row['Luz'] == 1 and row['Gas'] == 1: return 'Dual'
    elif row['Luz'] == 1: return 'Solo Luz'
    elif row['Gas'] == 1: return 'Solo Gas'
    else: return 'Sin Producto'

prod_orig['Clas_Suministro'] = prod_orig.apply(class_suministro, axis=1)
dist_orig = prod_orig['Clas_Suministro'].value_counts().to_dict()

# Distribución en la consolidada
dist_cons = df_clientes_f5['Clasificacion_Suministro'].value_counts().to_dict()

print("Distribución Original (Recalculada):", dist_orig)
print("Distribución Consolidada:", dist_cons)

check_2_passed = dist_orig == dist_cons
print(f"¿Comprobación 2 superada? (Coincidencia exacta): {'SÍ' if check_2_passed else 'NO'}")

results_summary.append({
    'Comprobación': '2. Distribución de Tipo de Suministro',
    'Esperado': str(dist_orig),
    'Observado': str(dist_cons),
    'Estado': 'Superado' if check_2_passed else 'Fallo'
})

=== 2. COMPARACIÓN DE LA DISTRIBUCIÓN DE PRODUCTOS (PRE VS POST) ===
Distribución Original (Recalculada): {'Solo Luz': 366953, 'Dual': 111861, 'Solo Gas': 20049}
Distribución Consolidada: {'Solo Luz': 366953, 'Dual': 111861, 'Solo Gas': 20049}
¿Comprobación 2 superada? (Coincidencia exacta): SÍ


In [ ]:
print("=== 3. ANÁLISIS DE LOS 290 CONTRATOS COMPARTIDOS Y RECONCILIACIÓN ===")

# Identificar contratos con múltiples clientes en origen
clientes_por_contrato = df.groupby('Contrato')['Num Cliente'].nunique()
contratos_compartidos = clientes_por_contrato[clientes_por_contrato > 1]

num_compartidos = len(contratos_compartidos)
print(f"- Contratos asociados a varios clientes distintos en origen: {num_compartidos} (Esperado: 290)")

# Mostrar distribución de cuántos clientes distintos tiene cada uno de estos identificadores
print("Distribución del número de clientes por contrato compartido:")
print(contratos_compartidos.value_counts())

# Comprobar la diferencia de reconciliación
suma_contratos_por_cliente = df_clientes_f5['Num_Contratos'].sum()
contratos_unicos_orig = df['Contrato'].nunique()
diferencia_calculada = suma_contratos_por_cliente - contratos_unicos_orig

print(f"- Suma de contratos por cliente: {suma_contratos_por_cliente}")
print(f"- Contratos únicos globales originales: {contratos_unicos_orig}")
print(f"- Diferencia matemática: {diferencia_calculada} (Esperado: 290)")

check_3_passed = (num_compartidos == 290) and (diferencia_calculada == 290)
print(f"¿Comprobación 3 superada?: {'SÍ' if check_3_passed else 'NO'}")

results_summary.append({
    'Comprobación': '3. Reconciliación de 290 contratos compartidos',
    'Esperado': '290 compartidos, Diferencia = 290',
    'Observado': f'{num_compartidos} compartidos, Diferencia = {diferencia_calculada}',
    'Estado': 'Superado' if check_3_passed else 'Fallo'
})

=== 3. ANÁLISIS DE LOS 290 CONTRATOS COMPARTIDOS Y RECONCILIACIÓN ===
- Contratos asociados a varios clientes distintos en origen: 290 (Esperado: 290)
Distribución del número de clientes por contrato compartido:
Num Cliente
2    290
Name: count, dtype: int64
- Suma de contratos por cliente: 577503
- Contratos únicos globales originales: 577213
- Diferencia matemática: 290 (Esperado: 290)
¿Comprobación 3 superada?: SÍ


In [ ]:
print("=== 4. COMPROBACIÓN DE CONFLICTOS DE TIPO CLIENTE ===")

# Recalcular clientes con discrepancias
tipos_por_cliente = df.groupby('Num Cliente')['Tipo Cliente'].nunique()
clientes_discrepantes_recalc = (tipos_por_cliente > 1).sum()

# Obtener del indicador de la tabla consolidada
clientes_conflicto_indicador = df_clientes_f5['Conflicto_Tipo_Cliente'].sum()

print(f"- Clientes con discrepancias recalculado: {clientes_discrepantes_recalc} (Esperado: 963)")
print(f"- Clientes marcados con flag de conflicto en la consolidada: {clientes_conflicto_indicador} (Esperado: 963)")

check_4_passed = clientes_discrepantes_recalc == clientes_conflicto_indicador
print(f"¿Comprobación 4 superada?: {'SÍ' if check_4_passed else 'NO'}")

results_summary.append({
    'Comprobación': '4. Coherencia del Flag de Conflicto',
    'Esperado': f'{clientes_discrepantes_recalc} clientes con flag',
    'Observado': f'{clientes_conflicto_indicador} clientes con flag',
    'Estado': 'Superado' if check_4_passed else 'Fallo'
})

=== 4. COMPROBACIÓN DE CONFLICTOS DE TIPO CLIENTE ===
- Clientes con discrepancias recalculado: 963 (Esperado: 963)
- Clientes marcados con flag de conflicto en la consolidada: 963 (Esperado: 963)
¿Comprobación 4 superada?: SÍ


In [ ]:
print("=== 5. TABLA RESUMEN DE EVALUACIÓN FINAL ===\n")
df_evaluacion = pd.DataFrame(results_summary)
display(df_evaluacion)

=== 5. TABLA RESUMEN DE EVALUACIÓN FINAL ===



,Comprobación,Esperado,Observado,Estado
0,1. Unicidad y Nulos de Num Cliente,"498863 filas, 0 duplicados, 0 nulos","498863 filas, 0 duplicados, 0 nulos",Superado
1,2. Distribución de Tipo de Suministro,"{'Solo Luz': 366953, 'Dual': 111861, 'Solo Gas...","{'Solo Luz': 366953, 'Dual': 111861, 'Solo Gas...",Superado
2,3. Reconciliación de 290 contratos compartidos,"290 compartidos, Diferencia = 290","290 compartidos, Diferencia = 290",Superado
3,4. Coherencia del Flag de Conflicto,963 clientes con flag,963 clientes con flag,Superado


### FASE 5A.2: VALIDACIÓN DE FECHAS Y CÁLCULO DE ANTIGÜEDAD

En esta sección validamos temporalmente la variable `Data Criação` y calculamos de forma segura la antigüedad comercial del cliente (`Antiguedad_Meses`) tomando el 24 de junio de 2026 como punto de corte, guardando los resultados en una copia de `df_clientes_f5`.

In [ ]:
# 1. Validación de la calidad de 'Data Criação' en el origen (df)
fecha_corte = pd.to_datetime('2026-06-24')

# Aseguramos formato datetime de la columna original en una serie de análisis
creacion_temp = pd.to_datetime(df['Data CriaÃ§Ã£o'], errors='coerce')

nulos_creacion = creacion_temp.isnull().sum()
fechas_futuras = (creacion_temp > fecha_corte).sum()

print("=== 1. VALIDACIÓN DE CALIDAD DE 'Data Criação' ===")
print(f"- Registros con fechas nulas o inválidas: {nulos_creacion}")
print(f"- Registros con fechas posteriores al corte (24-Jun-2026): {fechas_futuras}")

=== 1. VALIDACIÓN DE CALIDAD DE 'Data Criação' ===
- Registros con fechas nulas o inválidas: 0
- Registros con fechas posteriores al corte (24-Jun-2026): 0


In [ ]:
# 2. Cálculo de la Fecha Mínima por Cliente y Antigüedad
print("=== 2. CÁLCULO DE ANTIGÜEDAD A NIVEL DE CLIENTE ===")

# Creamos una copia de trabajo de df_clientes_f5 para no sobrescribir
df_clientes_f5_ant = df_clientes_f5.copy()

# Calculamos la fecha de creación mínima por cliente
fechas_minimas = df.groupby('Num Cliente')['Data CriaÃ§Ã£o'].min().reset_index()
fechas_minimas.rename(columns={'Data CriaÃ§Ã£o': 'Fecha_Primera_Creacion'}, inplace=True)
fechas_minimas['Fecha_Primera_Creacion'] = pd.to_datetime(fechas_minimas['Fecha_Primera_Creacion'], errors='coerce')

# Fusionamos en nuestra copia de la tabla consolidada
df_clientes_f5_ant = df_clientes_f5_ant.merge(fechas_minimas, on='Num Cliente', how='left')

# Fórmula de antigüedad en meses:
# Antiguedad_Meses = (Año_Referencia - Año_Creacion) * 12 + (Mes_Referencia - Mes_Creacion)
# Si hay días remanentes negativos o para ser precisos con punto flotante:
df_clientes_f5_ant['Antiguedad_Meses'] = (
    (fecha_corte - df_clientes_f5_ant['Fecha_Primera_Creacion']).dt.days / 30.4375
).apply(lambda x: np.maximum(0.0, x) if pd.notnull(x) else np.nan)

# Comprobación de nulos en el cálculo
nulos_fecha_min = df_clientes_f5_ant['Fecha_Primera_Creacion'].isnull().sum()
print(f"- Clientes cuya fecha mínima de creación no pudo determinarse (nulos): {nulos_fecha_min}")
print(f"- Número de filas de la tabla con antigüedad: {len(df_clientes_f5_ant)} (Esperado: {len(df_clientes_f5)})")

=== 2. CÁLCULO DE ANTIGÜEDAD A NIVEL DE CLIENTE ===
- Clientes cuya fecha mínima de creación no pudo determinarse (nulos): 0
- Número de filas de la tabla con antigüedad: 498863 (Esperado: 498863)


In [ ]:
print("=== 3. ESTADÍSTICOS DESCRIPTIVOS DE ANTIGUEDAD_MESES ===\n")
stats_antiguedad = df_clientes_f5_ant['Antiguedad_Meses'].describe()
display(stats_antiguedad)

=== 3. ESTADÍSTICOS DESCRIPTIVOS DE ANTIGUEDAD_MESES ===



,Antiguedad_Meses
count,498863.000000
mean,38.070894
std,35.539588
min,0.000000
25%,8.509240
50%,26.611910
75%,56.640657
max,131.482546


In [ ]:
# 2. Consolidación de Potencia Eléctrica y Escalón de Consumo de Gas a nivel de cliente
print("=== 2. PROCESAMIENTO Y AGREGACIÓN POR CLIENTE ===")

# Para potencia eléctrica: buscamos el valor máximo registrado por cliente (excluyendo '-')
df_potencia_valida = df[df['Potencia'] != '-'].copy()
potencia_cliente = df_potencia_valida.groupby('Num Cliente')['Potencia'].max().reset_index()
potencia_cliente.rename(columns={'Potencia': 'Potencia_Maxima'}, inplace=True)

# Para escalón de consumo: buscamos el valor máximo registrado por cliente (excluyendo '-')
escalao_col = 'EscalÃ£o Consumo' if 'EscalÃ£o Consumo' in df.columns else 'Escalão Consumo'
df_escalao_valido = df[df[escalao_col] != '-'].copy()
escalao_cliente = df_escalao_valido.groupby('Num Cliente')[escalao_col].max().reset_index()
escalao_cliente.rename(columns={escalao_col: 'Escalao_Consumo_Gas'}, inplace=True)

# Fusionamos estos atributos en nuestra tabla de trabajo con antigüedad 'df_clientes_f5_ant'
df_clientes_f5_ant = df_clientes_f5_ant.merge(potencia_cliente, on='Num Cliente', how='left')
df_clientes_f5_ant = df_clientes_f5_ant.merge(escalao_cliente, on='Num Cliente', how='left')

# Rellenamos con 'No Aplica / Indet.' donde corresponda para mantener consistencia de negocio
df_clientes_f5_ant['Potencia_Maxima'] = df_clientes_f5_ant['Potencia_Maxima'].fillna('No Aplica / Indet.')
df_clientes_f5_ant['Escalao_Consumo_Gas'] = df_clientes_f5_ant['Escalao_Consumo_Gas'].fillna('No Aplica / Indet.')

print(f"- Registros en la tabla final consolidada: {len(df_clientes_f5_ant)} (Esperado: 498863)")
print(f"- ¿Hay duplicados en Num Cliente?: {df_clientes_f5_ant['Num Cliente'].duplicated().any()}")

print("\n=== Distribución de Potencia Máxima por Cliente ===")
display(df_clientes_f5_ant['Potencia_Maxima'].value_counts())

print("\n=== Distribución de Escalón de Consumo de Gas por Cliente ===")
display(df_clientes_f5_ant['Escalao_Consumo_Gas'].value_counts())

=== 2. PROCESAMIENTO Y AGREGACIÓN POR CLIENTE ===
- Registros en la tabla final consolidada: 498863 (Esperado: 498863)
- ¿Hay duplicados en Num Cliente?: False

=== Distribución de Potencia Máxima por Cliente ===


,count
Potencia_Maxima,
EDE130,173265
EDE140,136268
EDE133,73808
EDE137,39454
EDE150,27931
No Aplica / Indet.,20049
EDE160,10132
EDE180,8030
EDE110,4099



=== Distribución de Escalón de Consumo de Gas por Cliente ===


,count
Escalao_Consumo_Gas,
No Aplica / Indet.,366953
BP < (0 - 220),93466
BP < (221 - 500),32858
BP < (500 - 1000),4532
BP < (1001 - 10000),1054


In [ ]:
# VERIFICACIÓN FINAL - FASE 5A.2
print("=== 1. FÓRMULA DE ANTIGÜEDAD MESES ===")
print("Fórmula aplicada: (Fecha Corte - Fecha_Primera_Creacion).dt.days / 30.4375")
print("Explicación: Se basa en la diferencia en días naturales dividida por el mes medio de la Tierra (365.25 días / 12 meses = 30.4375 días) para máxima precisión decimal.\n")

print("=== 2. VERIFICACIÓN DE CORRESPONDENCIA DE FECHA MÍNIMA ===")
# Recalculamos directamente del origen para verificar con la tabla consolidada
fechas_min_origen = df.groupby('Num Cliente')['Data CriaÃ§Ã£o'].min()
coincide_fechas = df_clientes_f5_ant.set_index('Num Cliente')['Fecha_Primera_Creacion'].equals(fechas_min_origen)
print(f"¿Coincide exactamente la fecha mínima para cada cliente entre el origen y el destino?: {coincide_fechas}\n")

print("=== 3. VERIFICACIÓN DE ANTIGÜEDADES NEGATIVAS PREVIAS ===")
# Calculamos la diferencia bruta en días sin aplicar np.maximum
dias_brutos = (fecha_corte - df_clientes_f5_ant['Fecha_Primera_Creacion']).dt.days
antiguedades_negativas = (dias_brutos < 0).sum()
print(f"Número de registros con antigüedad negativa teórica (días < 0) antes de aplicar límites: {antiguedades_negativas}\n")

print("=== 4. INTEGRIDAD ESTRUCTURAL Y VALORES NULOS ===")
filas_cons = len(df_clientes_f5_ant)
unicos_cons = df_clientes_f5_ant['Num Cliente'].nunique()
nulos_fecha = df_clientes_f5_ant['Fecha_Primera_Creacion'].isnull().sum()
nulos_ant = df_clientes_f5_ant['Antiguedad_Meses'].isnull().sum()

print(f"- Número de filas: {filas_cons} (Esperado: 498863)")
print(f"- ¿Num Cliente es clave primaria única?: {filas_cons == unicos_cons}")
print(f"- Nulos en 'Fecha_Primera_Creacion': {nulos_fecha}")
print(f"- Nulos en 'Antiguedad_Meses': {nulos_ant}\n")

print("=== 5. ESTADÍSTICOS DESCRIPTIVOS DE ANTIGUEDAD_MESES ===")
display(df_clientes_f5_ant['Antiguedad_Meses'].describe())

=== 1. FÓRMULA DE ANTIGÜEDAD MESES ===
Fórmula aplicada: (Fecha Corte - Fecha_Primera_Creacion).dt.days / 30.4375
Explicación: Se basa en la diferencia en días naturales dividida por el mes medio de la Tierra (365.25 días / 12 meses = 30.4375 días) para máxima precisión decimal.

=== 2. VERIFICACIÓN DE CORRESPONDENCIA DE FECHA MÍNIMA ===
¿Coincide exactamente la fecha mínima para cada cliente entre el origen y el destino?: True

=== 3. VERIFICACIÓN DE ANTIGÜEDADES NEGATIVAS PREVIAS ===
Número de registros con antigüedad negativa teórica (días < 0) antes de aplicar límites: 0

=== 4. INTEGRIDAD ESTRUCTURAL Y VALORES NULOS ===
- Número de filas: 498863 (Esperado: 498863)
- ¿Num Cliente es clave primaria única?: True
- Nulos en 'Fecha_Primera_Creacion': 0
- Nulos en 'Antiguedad_Meses': 0

=== 5. ESTADÍSTICOS DESCRIPTIVOS DE ANTIGUEDAD_MESES ===


,Antiguedad_Meses
count,498863.000000
mean,38.070894
std,35.539588
min,0.000000
25%,8.509240
50%,26.611910
75%,56.640657
max,131.482546


### FASE 5B.1: DIAGNÓSTICO DE VARIABLES COMERCIALES POR PRODUCTO
En esta sección realizamos un perfilado estricto y un análisis de consistencia interna de las variables `Potencia` (para electricidad) y `Escalão Consumo` (para gas), evaluando su comportamiento en el dataset de origen `df` y la variabilidad de estos atributos por cada cliente.

In [ ]:
import pandas as pd
import numpy as np

# Identificación de la columna correcta de escalón de consumo en pandas
escalao_col = 'EscalÃ£o Consumo' if 'EscalÃ£o Consumo' in df.columns else 'Escalão Consumo'

print("=== 1. DISTRIBUCIÓN DE POTENCIA ELÉCTRICA (SÓLO ELECTRICIDAD) ===")
# Filtramos registros de electricidad
df_elec = df[df['Prod'].str.contains('EB|Elect', na=False, case=False)].copy()
total_elec_rows = len(df_elec)

potencia_counts = df_elec['Potencia'].value_counts(dropna=False)
potencia_rel = df_elec['Potencia'].value_counts(dropna=False, normalize=True) * 100

dist_potencia = pd.DataFrame({
    'Frecuencia Absoluta': potencia_counts,
    'Frecuencia Relativa (%)': potencia_rel
})
display(dist_potencia)
print(f"Nulos reales (NaN) en Potencia: {df_elec['Potencia'].isnull().sum()}")
print(f"Guiones '-' en Potencia: {(df_elec['Potencia'] == '-').sum()}\n")


print("=== 2. DISTRIBUCIÓN DE ESCALONES DE CONSUMO DE GAS (SÓLO GAS) ===")
# Filtramos registros de gas
df_gas = df[df['Prod'].str.contains('GP|G\u00e1s|G\u00f3s', na=False, case=False)].copy()
total_gas_rows = len(df_gas)

escalao_counts = df_gas[escalao_col].value_counts(dropna=False)
escalao_rel = df_gas[escalao_col].value_counts(dropna=False, normalize=True) * 100

dist_escalao = pd.DataFrame({
    'Frecuencia Absoluta': escalao_counts,
    'Frecuencia Relativa (%)': escalao_rel
})
display(dist_escalao)
print(f"Nulos reales (NaN) en Escal\u00f3n Consumo: {df_gas[escalao_col].isnull().sum()}")
print(f"Guiones '-' en Escal\u00f3n Consumo: {(df_gas[escalao_col] == '-').sum()}\n")


print("=== 3. VARIABILIDAD POR CLIENTE ===")
# Potencia por cliente en contratos eléctricos
pot_por_cliente_elec = df_elec.groupby('Num Cliente')['Potencia'].nunique()
clientes_con_multi_pot = (pot_por_cliente_elec > 1).sum()
clientes_con_una_pot = (pot_por_cliente_elec == 1).sum()

print(f"Clientes con contratos eléctricos analizados: {len(pot_por_cliente_elec)}")
print(f" - Clientes con un \u00fanico valor de Potencia: {clientes_con_una_pot} ({(clientes_con_una_pot / len(pot_por_cliente_elec))*100:.3f}%)")
print(f" - Clientes con m\u00faltiples valores de Potencia: {clientes_con_multi_pot} ({(clientes_con_multi_pot / len(pot_por_cliente_elec))*100:.3f}%)")

# Escalón de consumo por cliente en contratos de gas
esc_por_cliente_gas = df_gas.groupby('Num Cliente')[escalao_col].nunique()
clientes_con_multi_esc = (esc_por_cliente_gas > 1).sum()
clientes_con_un_esc = (esc_por_cliente_gas == 1).sum()

print(f"\nClientes con contratos de gas analizados: {len(esc_por_cliente_gas)}")
print(f" - Clientes con un \u00fanico valor de Escal\u00f3n Consumo: {clientes_con_un_esc} ({(clientes_con_un_esc / len(esc_por_cliente_gas))*100:.3f}%)")
print(f" - Clientes con m\u00faltiples valores de Escal\u00f3n Consumo: {clientes_con_multi_esc} ({(clientes_con_multi_esc / len(esc_por_cliente_gas))*100:.3f}%)\n")


print("=== 4. DIFERENCIAS ENTRE CONTRATOS DEL MISMO PRODUCTO ===")
# Clientes eléctricos con múltiples contratos y valores distintos de Potencia
df_multi_elec = df_elec[df_elec['Num Cliente'].isin(pot_por_cliente_elec[pot_por_cliente_elec > 1].index)]
print(f"Clientes afectados por discordancia de Potencia el\u00e9ctrica: {df_multi_elec['Num Cliente'].nunique()}")
print(f"Contratos totales asociados a estos clientes: {df_multi_elec['Contrato'].nunique()}")

# Muestra anonimizada de ejemplo de conflicto de Potencia
if not df_multi_elec.empty:
    ejemplo_pot_ids = df_multi_elec['Num Cliente'].unique()[:3]
    print("\nMuestra representativa de discordancia de Potencia (por Cliente):")
    display(df_multi_elec[df_multi_elec['Num Cliente'].isin(ejemplo_pot_ids)][['Num Cliente', 'Contrato', 'Potencia', 'Est Contrato']].sort_values('Num Cliente'))

# Clientes de gas con múltiples contratos y valores distintos de Escalón
df_multi_gas = df_gas[df_gas['Num Cliente'].isin(esc_por_cliente_gas[esc_por_cliente_gas > 1].index)]
print(f"\nClientes afectados por discordancia de Escal\u00f3n de Consumo: {df_multi_gas['Num Cliente'].nunique()}")
print(f"Contratos totales asociados a estos clientes: {df_multi_gas['Contrato'].nunique()}")

# Muestra anonimizada de ejemplo de conflicto de Escalón de consumo
if not df_multi_gas.empty:
    ejemplo_esc_ids = df_multi_gas['Num Cliente'].unique()[:3]
    print("\nMuestra de discordancia de Escal\u00f3n de Consumo de gas:")
    display(df_multi_gas[df_multi_gas['Num Cliente'].isin(ejemplo_esc_ids)][['Num Cliente', 'Contrato', escalao_col, 'Est Contrato']].sort_values('Num Cliente'))

=== 1. DISTRIBUCIÓN DE POTENCIA ELÉCTRICA (SÓLO ELECTRICIDAD) ===


,Frecuencia Absoluta,Frecuencia Relativa (%)
Potencia,,
EDE130,197026,37.186013
EDE140,147699,27.876204
EDE133,78457,14.807706
EDE137,41832,7.895229
EDE150,30317,5.721927
EDE160,11426,2.156504
EDE180,8842,1.668809
EDE110,7144,1.348334
EDE170,4024,0.759476


Nulos reales (NaN) en Potencia: 0
Guiones '-' en Potencia: 0

=== 2. DISTRIBUCIÓN DE ESCALONES DE CONSUMO DE GAS (SÓLO GAS) ===


,Frecuencia Absoluta,Frecuencia Relativa (%)
EscalÃ£o Consumo,,
BP < (0 - 220),95779,71.194214
BP < (221 - 500),33098,24.602325
BP < (500 - 1000),4560,3.389528
BP < (1001 - 10000),1095,0.813933


Nulos reales (NaN) en Escalón Consumo: 0
Guiones '-' en Escalón Consumo: 0

=== 3. VARIABILIDAD POR CLIENTE ===
Clientes con contratos eléctricos analizados: 478814
 - Clientes con un único valor de Potencia: 454870 (94.999%)
 - Clientes con múltiples valores de Potencia: 23944 (5.001%)

Clientes con contratos de gas analizados: 131910
 - Clientes con un único valor de Escalón Consumo: 131076 (99.368%)
 - Clientes con múltiples valores de Escalón Consumo: 834 (0.632%)

=== 4. DIFERENCIAS ENTRE CONTRATOS DEL MISMO PRODUCTO ===
Clientes afectados por discordancia de Potencia eléctrica: 23944
Contratos totales asociados a estos clientes: 59960

Muestra representativa de discordancia de Potencia (por Cliente):


,Num Cliente,Contrato,Potencia,Est Contrato
1910,321244,39937601108,EDE140,14 - Contrato Activado nÃ£o Factur.
162,321244,770997105,EDE130,14 - Contrato Activado nÃ£o Factur.
134072,321244,752675104,EDE130,12 - Contrato em Vigor
662723,321244,1567003101,EDE130,15 - Contrato Pendente de ActivaÃ§Ã£o
2608,579553,1170755103,EDE150,14 - Contrato Activado nÃ£o Factur.
2607,579553,1165234103,EDE130,14 - Contrato Activado nÃ£o Factur.
151,579553,1165243104,EDE137,14 - Contrato Activado nÃ£o Factur.
32,82054635,1568508101,EDE140,14 - Contrato Activado nÃ£o Factur.
662238,82054635,532608102,EDE133,15 - Contrato Pendente de ActivaÃ§Ã£o



Clientes afectados por discordancia de Escalón de Consumo: 834
Contratos totales asociados a estos clientes: 1975

Muestra de discordancia de Escalón de Consumo de gas:


,Num Cliente,Contrato,EscalÃ£o Consumo,Est Contrato
1212,647869,1570535101,BP < (0 - 220),14 - Contrato Activado nÃ£o Factur.
1211,647869,1570518101,BP < (500 - 1000),14 - Contrato Activado nÃ£o Factur.
692,1209489,1569525101,BP < (0 - 220),14 - Contrato Activado nÃ£o Factur.
78303,1209489,1509990101,BP < (221 - 500),12 - Contrato em Vigor
3039,1221322,1569344101,BP < (0 - 220),14 - Contrato Activado nÃ£o Factur.
3931,1221322,1568751101,BP < (0 - 220),14 - Contrato Activado nÃ£o Factur.
3935,1221322,1568803101,BP < (0 - 220),14 - Contrato Activado nÃ£o Factur.
1206,1221322,1570080101,BP < (0 - 220),14 - Contrato Activado nÃ£o Factur.
4848,1221322,1567500101,BP < (0 - 220),14 - Contrato Activado nÃ£o Factur.
4849,1221322,1567501101,BP < (0 - 220),14 - Contrato Activado nÃ£o Factur.


In [ ]:
# 1. Verificación de Estados de Contrato analizados y filtros
print("=== 1. COMPROBACIÓN DE FILTROS Y ESTADOS DE CONTRATO ===")
estados_luz = df[df['Prod'].str.contains('EB|Elect', na=False, case=False)]['Est Contrato'].value_counts()
estados_gas = df[df['Prod'].str.contains('GP|G\u00e1s|G\u00f3s', na=False, case=False)]['Est Contrato'].value_counts()

print("Estados incluidos en contratos de Electricidad:")
display(estados_luz)
print("\nEstados incluidos en contratos de Gas:")
display(estados_gas)

# 2. Verificación de Clientes Únicos por Producto
print("\n=== 2. COMPROBACIÓN DE CLIENTES ÚNICOS ===")
clientes_luz_unicos = df[df['Prod'].str.contains('EB|Elect', na=False, case=False)]['Num Cliente'].nunique()
clientes_gas_unicos = df[df['Prod'].str.contains('GP|G\u00e1s|G\u00f3s', na=False, case=False)]['Num Cliente'].nunique()
print(f"Clientes únicos con Electricidad: {clientes_luz_unicos}")
print(f"Clientes únicos con Gas: {clientes_gas_unicos}")

# 3. Recalculo de Variabilidad de Atributos (excluyendo '-')
print("\n=== 3. RECALCULO DE VARIABILIDAD Y VALORES NO VÁLIDOS ===")
# Luz
pot_por_cli = df_elec.groupby('Num Cliente')['Potencia'].nunique()
print(f"Clientes Luz con 1 potencia única: {(pot_por_cli == 1).sum()} ({(pot_por_cli == 1).sum() / len(pot_por_cli)*100:.3f}%)")
print(f"Clientes Luz con >1 potencia distinta: {(pot_por_cli > 1).sum()} ({(pot_por_cli > 1).sum() / len(pot_por_cli)*100:.3f}%)")

# Gas
esc_por_cli = df_gas.groupby('Num Cliente')[escalao_col].nunique()
print(f"Clientes Gas con 1 escalón único: {(esc_por_cli == 1).sum()} ({(esc_por_cli == 1).sum() / len(esc_por_cli)*100:.3f}%)")
print(f"Clientes Gas con >1 escalón distinto: {(esc_por_cli > 1).sum()} ({(esc_por_cli > 1).sum() / len(esc_por_cli)*100:.3f}%)")

=== 1. COMPROBACIÓN DE FILTROS Y ESTADOS DE CONTRATO ===
Estados incluidos en contratos de Electricidad:


,count
Est Contrato,
12 - Contrato em Vigor,510922
14 - Contrato Activado nÃ£o Factur.,17341
15 - Contrato Pendente de ActivaÃ§Ã£o,1576



Estados incluidos en contratos de Gas:


,count
Est Contrato,
12 - Contrato em Vigor,128653
14 - Contrato Activado nÃ£o Factur.,5271
15 - Contrato Pendente de ActivaÃ§Ã£o,608



=== 2. COMPROBACIÓN DE CLIENTES ÚNICOS ===
Clientes únicos con Electricidad: 478814
Clientes únicos con Gas: 131910

=== 3. RECALCULO DE VARIABILIDAD Y VALORES NO VÁLIDOS ===
Clientes Luz con 1 potencia única: 454870 (94.999%)
Clientes Luz con >1 potencia distinta: 23944 (5.001%)
Clientes Gas con 1 escalón único: 131076 (99.368%)
Clientes Gas con >1 escalón distinto: 834 (0.632%)


In [ ]:
import pandas as pd
import numpy as np

# Identificación de la columna correcta de escalón de consumo
escalao_col = 'EscalÃ£o Consumo' if 'EscalÃ£o Consumo' in df.columns else 'Escalão Consumo'

# 1. Comprobar si las diferencias ocurren entre contratos distintos del mismo producto
print("=== 1. DIFERENCIAS DE ATRIBUTOS ENTRE CONTRATOS DISTINTOS DEL MISMO PRODUCTO ===")

# Clientes de electricidad con múltiple potencia
cli_multi_pot = pot_por_cliente_elec[pot_por_cliente_elec > 1].index
df_multi_pot = df_elec[df_elec['Num Cliente'].isin(cli_multi_pot)]

# Clientes de gas con múltiple escalón
cli_multi_esc = esc_por_cliente_gas[esc_por_cliente_gas > 1].index
df_multi_esc = df_gas[df_gas['Num Cliente'].isin(cli_multi_esc)]

# Verificación: contratos por cliente para estos casos
contratos_por_cli_pot = df_multi_pot.groupby('Num Cliente')['Contrato'].nunique()
contratos_por_cli_esc = df_multi_esc.groupby('Num Cliente')['Contrato'].nunique()

print(f"Clientes Luz con >1 potencia: {len(cli_multi_pot)}")
print(f" - Mínimo de contratos distintos por cliente: {contratos_por_cli_pot.min()}")
print(f" - Máximo de contratos distintos por cliente: {contratos_por_cli_pot.max()}")
print(f" - Media de contratos distintos por cliente: {contratos_por_cli_pot.mean():.2f}")

print(f"\nClientes Gas con >1 escalón: {len(cli_multi_esc)}")
print(f" - Mínimo de contratos distintos por cliente: {contratos_por_cli_esc.min()}")
print(f" - Máximo de contratos distintos por cliente: {contratos_por_cli_esc.max()}")
print(f" - Media de contratos distintos por cliente: {contratos_por_cli_esc.mean():.2f}")

# Muestra anonimizada de ejemplos (Luz)
print("\n--- Ejemplos de clientes con múltiples potencias (Luz) ---")
ejemplos_luz = []
for cli in list(cli_multi_pot)[:3]:
    sub = df_multi_pot[df_multi_pot['Num Cliente'] == cli]
    ejemplos_luz.append(sub[['Num Cliente', 'Contrato', 'Potencia', 'Est Contrato']])
display(pd.concat(ejemplos_luz))

# Muestra anonimizada de ejemplos (Gas)
print("\n--- Ejemplos de clientes con múltiples escalones de consumo (Gas) ---")
ejemplos_gas = []
for cli in list(cli_multi_esc)[:3]:
    sub = df_multi_esc[df_multi_esc['Num Cliente'] == cli]
    ejemplos_gas.append(sub[['Num Cliente', 'Contrato', escalao_col, 'Est Contrato']])
display(pd.concat(ejemplos_gas))

=== 1. DIFERENCIAS DE ATRIBUTOS ENTRE CONTRATOS DISTINTOS DEL MISMO PRODUCTO ===
Clientes Luz con >1 potencia: 23944
 - Mínimo de contratos distintos por cliente: 2
 - Máximo de contratos distintos por cliente: 121
 - Media de contratos distintos por cliente: 2.50

Clientes Gas con >1 escalón: 834
 - Mínimo de contratos distintos por cliente: 2
 - Máximo de contratos distintos por cliente: 52
 - Media de contratos distintos por cliente: 2.37

--- Ejemplos de clientes con múltiples potencias (Luz) ---


,Num Cliente,Contrato,Potencia,Est Contrato
458173,166,768316101,EDE150,12 - Contrato em Vigor
594027,166,30489001103,EDE130,12 - Contrato em Vigor
596139,166,30488401102,EDE150,12 - Contrato em Vigor
660335,204,95025801101,EDE140,12 - Contrato em Vigor
660339,204,204101,EDE130,12 - Contrato em Vigor
640587,309,310102,EDE180,12 - Contrato em Vigor
640702,309,33730301102,EDE130,12 - Contrato em Vigor



--- Ejemplos de clientes con múltiples escalones de consumo (Gas) ---


,Num Cliente,Contrato,EscalÃ£o Consumo,Est Contrato
639130,102,308101,BP < (0 - 220),12 - Contrato em Vigor
639254,102,101101,BP < (221 - 500),12 - Contrato em Vigor
368701,5102,22014001102,BP < (221 - 500),12 - Contrato em Vigor
372794,5102,1006802101,BP < (221 - 500),12 - Contrato em Vigor
636963,5102,5009101,BP < (0 - 220),12 - Contrato em Vigor
182577,5987,1401883101,BP < (221 - 500),12 - Contrato em Vigor
484728,5987,38896001101,BP < (0 - 220),12 - Contrato em Vigor


In [ ]:
print("=== 2. ESTADOS DE CONTRATO (EST CONTRATO) EN CLIENTES CON VARIABILIDAD ===")

def analizar_estados_grupo(df_subset):
    # Agrupamos por cliente y analizamos el set de sus estados de contrato
    estados_por_cli = df_subset.groupby('Num Cliente')['Est Contrato'].apply(set)

    all_vigor = 0
    mixed_or_other = 0

    for estados in estados_por_cli:
        # Limpiamos espacios y caracteres
        estados_clean = {str(e).strip() for e in estados}
        # Si el único estado es 'em Vigor' (código 12)
        if len(estados_clean) == 1 and any('12' in e or 'Vigor' in e for e in estados_clean):
            all_vigor += 1
        else:
            mixed_or_other += 1

    return all_vigor, mixed_or_other, estados_por_cli.value_counts()

vigor_pot, mixtos_pot, dist_estados_pot = analizar_estados_grupo(df_multi_pot)
vigor_esc, mixtos_esc, dist_estados_esc = analizar_estados_grupo(df_multi_esc)

print("Clientes con múltiples Potencias (Luz):")
print(f" - Todos sus contratos 'En Vigor': {vigor_pot} ({vigor_pot/len(cli_multi_pot)*100:.2f}%)")
print(f" - Con contratos activados no facturados, pendientes o mixtos: {mixtos_pot} ({mixtos_pot/len(cli_multi_pot)*100:.2f}%)")
print(" - Frecuencia de combinaciones de estados:")
print(dist_estados_pot)

print("\nClientes con múltiples Escalones de Consumo (Gas):")
print(f" - Todos sus contratos 'En Vigor': {vigor_esc} ({vigor_esc/len(cli_multi_esc)*100:.2f}%)")
print(f" - Con contratos activados no facturados, pendientes o mixtos: {mixtos_esc} ({mixtos_esc/len(cli_multi_esc)*100:.2f}%)")
print(" - Frecuencia de combinaciones de estados:")
print(dist_estados_esc)

=== 2. ESTADOS DE CONTRATO (EST CONTRATO) EN CLIENTES CON VARIABILIDAD ===
Clientes con múltiples Potencias (Luz):
 - Todos sus contratos 'En Vigor': 22524 (94.07%)
 - Con contratos activados no facturados, pendientes o mixtos: 1420 (5.93%)
 - Frecuencia de combinaciones de estados:
Est Contrato
{12 - Contrato em Vigor}                                                                                22524
{12 - Contrato em Vigor, 14 - Contrato Activado nÃ£o Factur.}                                             704
{14 - Contrato Activado nÃ£o Factur.}                                                                     585
{15 - Contrato Pendente de ActivaÃ§Ã£o, 12 - Contrato em Vigor}                                            71
{15 - Contrato Pendente de ActivaÃ§Ã£o, 14 - Contrato Activado nÃ£o Factur.}                               32
{15 - Contrato Pendente de ActivaÃ§Ã£o}                                                                    22
{15 - Contrato Pendente de ActivaÃ§Ã£o, 12 

In [ ]:
print("=== 3. RECUENTOS DE CONTRATO ÚNICOS Y REGISTROS POR CONTRATO ===")

# Comprobar si el conteo de contratos afectados corresponde a IDs únicos de contrato
total_reg_pot = len(df_multi_pot)
unicos_con_pot = df_multi_pot['Contrato'].nunique()
total_reg_esc = len(df_multi_esc)
unicos_con_esc = df_multi_esc['Contrato'].nunique()

print("Para los clientes con múltiples Potencias (Luz):")
print(f" - Total registros en el subset: {total_reg_pot}")
print(f" - Contratos únicos en el subset: {unicos_con_pot}")

print("\nPara los clientes con múltiples Escalones (Gas):")
print(f" - Total registros en el subset: {total_reg_esc}")
print(f" - Contratos únicos en el subset: {unicos_con_esc}")

# Evaluar si un mismo cliente tiene más de un registro para un mismo contrato en el dataset global
print("\n--- Comprobación de multiplicidad de registros por combinación Cliente + Contrato (Dataset completo) ---")
registros_por_cliente_contrato = df.groupby(['Num Cliente', 'Contrato']).size()
max_reg = registros_por_cliente_contrato.max()
print(f"Máximo número de registros para un mismo 'Num Cliente' y 'Contrato': {max_reg}")

if max_reg > 1:
    print(f"Número de combinaciones Cliente-Contrato repetidas: {(registros_por_cliente_contrato > 1).sum()}")
    print("Muestra de contratos duplicados para un mismo cliente:")
    ejemplo_dup = registros_por_cliente_contrato[registros_por_cliente_contrato > 1].index[0]
    display(df[(df['Num Cliente'] == ejemplo_dup[0]) & (df['Contrato'] == ejemplo_dup[1])][['Num Cliente', 'Contrato', 'Prod', 'Est Contrato', 'Data CriaÃ§Ã£o']])
else:
    print("Confirmado: Cada combinación de Num Cliente y Contrato aparece exactamente una única vez en el dataset.")

=== 3. RECUENTOS DE CONTRATO ÚNICOS Y REGISTROS POR CONTRATO ===
Para los clientes con múltiples Potencias (Luz):
 - Total registros en el subset: 59960
 - Contratos únicos en el subset: 59960

Para los clientes con múltiples Escalones (Gas):
 - Total registros en el subset: 1975
 - Contratos únicos en el subset: 1975

--- Comprobación de multiplicidad de registros por combinación Cliente + Contrato (Dataset completo) ---
Máximo número de registros para un mismo 'Num Cliente' y 'Contrato': 4
Número de combinaciones Cliente-Contrato repetidas: 86866
Muestra de contratos duplicados para un mismo cliente:


,Num Cliente,Contrato,Prod,Est Contrato,Data CriaÃ§Ã£o
309477,21,22103,GP - GÃ¡s,12 - Contrato em Vigor,2024-08-30
309527,21,22103,EB - Elect. Baixa TensÃ£o,12 - Contrato em Vigor,2024-08-30


### FASE 5B.2: DISEÑO Y PLANIFICACIÓN DE LA CONSOLIDACIÓN COMERCIAL
En esta sección se confirma el estado actual de los dataframes de trabajo, se analizan los criterios técnicos para la agregación de variables comerciales (Potencia, Escalones de Consumo y Estados de Contrato) y se presenta el diseño lógico de la tabla de salida de cara a la segmentación del negocio.

In [ ]:
import pandas as pd
import numpy as np

print("=== 1. VERIFICACIÓN DE LAS DIMENSIONES Y CLAVES ===")
print(f"Dimensiones de 'df' (Origen): {df.shape[0]} filas, {df.shape[1]} columnas.")
print(f"Dimensiones de 'df_clientes_f5_ant' (Consolidado Temporal): {df_clientes_f5_ant.shape[0]} filas, {df_clientes_f5_ant.shape[1]} columnas.")

# Verificar clave primaria de df_clientes_f5_ant
es_pk = not df_clientes_f5_ant['Num Cliente'].duplicated().any()
print(f"¿'Num Cliente' es Clave Primaria única en 'df_clientes_f5_ant'?: {es_pk}")

print("\n=== 2. DETECTAR VARIABLES EXISTENTES EN df_clientes_f5_ant ===")
print(list(df_clientes_f5_ant.columns))

print("\n=== 3. COMPARATIVA DE DEFINICIÓN DE RECANTOS DE CONTRATO ===")
# Alternativa A: Contratos únicos (Distinct 'Contrato' IDs)
contratos_unicos_totales = df['Contrato'].nunique()
sum_contratos_unicos_por_cli = df.groupby('Num Cliente')['Contrato'].nunique().sum()

# Alternativa B: Combinaciones distintas Num Cliente + Contrato (Frecuencia de filas)
comb_cli_contrato = df.groupby(['Num Cliente', 'Contrato']).size().shape[0]

print(f"- Total de IDs de Contrato únicos a nivel global: {contratos_unicos_totales}")
print(f"- Suma de IDs de Contrato únicos calculados por cliente: {sum_contratos_unicos_por_cli}")
print(f"- Total de combinaciones únicas [Num Cliente + Contrato]: {comb_cli_contrato}")
print(f"- Diferencia entre combinaciones [Num Cliente + Contrato] y Contratos únicos agregados por cliente: {comb_cli_contrato - sum_contratos_unicos_por_cli} (Debe ser 0)")

=== 1. VERIFICACIÓN DE LAS DIMENSIONES Y CLAVES ===
Dimensiones de 'df' (Origen): 664371 filas, 36 columnas.
Dimensiones de 'df_clientes_f5_ant' (Consolidado Temporal): 498863 filas, 11 columnas.
¿'Num Cliente' es Clave Primaria única en 'df_clientes_f5_ant'?: True

=== 2. DETECTAR VARIABLES EXISTENTES EN df_clientes_f5_ant ===
['Num Cliente', 'Tiene_Luz', 'Tiene_Gas', 'Num_Productos', 'Clasificacion_Suministro', 'Num_Contratos', 'Conflicto_Tipo_Cliente', 'Fecha_Primera_Creacion', 'Antiguedad_Meses', 'Potencia_Maxima', 'Escalao_Consumo_Gas']

=== 3. COMPARATIVA DE DEFINICIÓN DE RECANTOS DE CONTRATO ===
- Total de IDs de Contrato únicos a nivel global: 577213
- Suma de IDs de Contrato únicos calculados por cliente: 577503
- Total de combinaciones únicas [Num Cliente + Contrato]: 577503
- Diferencia entre combinaciones [Num Cliente + Contrato] y Contratos únicos agregados por cliente: 0 (Debe ser 0)


### PROPUESTA METODOLÓGICA Y DISEÑO LÓGICO (F5B.2)

#### 1. Análisis de Recuento de Contratos (`Num_Contratos`)
La comparación matemática realizada en la celda anterior demuestra que:
* El recuento de **identificadores únicos de contrato** agregados por cliente da una suma de **577,503**.
* El recuento de combinaciones únicas de **`Num Cliente` + `Contrato`** da exactamente el mismo valor (**577,503**).

**Recomendación para el TFM:** Se recomienda utilizar la agregación basada en el recuento de contratos únicos por cliente (`df.groupby('Num Cliente')['Contrato'].nunique()`).
* **Justificación:** Mide con precisión cuántas relaciones contractuales independientes posee cada ficha de cliente. La discrepancia de $290$ observada respecto a los contratos únicos globales del dataset original se debe a los $290$ contratos que aparecen compartidos bajo más de un código de cliente. Contabilizarlo mediante `nunique()` a nivel cliente asegura que cada contrato sea imputado de forma correcta a la cartera de su respectivo cliente, reflejando su volumen comercial real.

---

#### 2. Reglas de Agregación Comerciales Propuestas (Sin Ejecución)

* **`Tiene_Luz` / `Tiene_Gas`:** Variables binarias ($1$ o $0$) basadas en si el cliente posee algún registro de producto de baja tensión (`EB`) o gas (`GP`).
* **`Num_Potencias_Distintas`:** `df[df['Prod'] == 'EB'].groupby('Num Cliente')['Potencia'].nunique()`. Mide la variabilidad técnica interna. Para clientes sin electricidad, se establece como $0$.
* **`Num_Escalones_Gas_Distintos`:** `df[df['Prod'] == 'GP'].groupby('Num Cliente')['Escalão Consumo'].nunique()`. Mide la variabilidad de consumo en gas. Para clientes sin gas, se establece como $0$.
* **`Potencia_Maxima` / `Escalao_Consumo_Gas`:** Preservan las dimensiones originales correspondientes para caracterizar el perfil energético. Se rellenan con `'No Aplica / Indet.'` en aquellos clientes que no dispongan del respectivo suministro.

---

#### 3. Tratamiento de Estados Contractuales y Valores Especiales

El dataset de origen cuenta con tres estados en `Est Contrato`:
1. `12 - Contrato em Vigor` (Vigente y facturable)
2. `14 - Contrato Activado não Factur.` (Activo pero sin emitir factura aún)
3. `15 - Contrato Pendente de Activação` (Pendiente de inicio de suministro)

##### Comparativa de Criterios de Selección

| Criterio | Ventajas | Desventajas | Recomendación de Uso |
| :--- | :--- | :--- | :--- |
| **Criterio 1: Considerar Todos los Estados** (Recomendado) | Ofrece una visión global y fidedigna de la cartera comercial del cliente. Permite identificar clientes híbridos con contratos activos y en proceso de instalación, clave para segmentar la propensión al abandono (*churn*) o captación reciente. | Puede distorsionar el análisis de consumo o potencia actuales si se asume que los contratos pendientes ya consumen recursos. | **Elegido para la caracterización**. Ofrece la foto completa de la base de datos sin pérdida de representatividad. |
| **Criterio 2: Filtrar Solo Contratos 'En Vigor'** | Garantiza que las potencias y tarifas analizadas corresponden exclusivamente a instalaciones activas y facturables en tiempo real. | Excluye un total de **24,796 registros** (estados 14 y 15) que representan clientes reales que acaban de entrar o cambiar de cartera, perdiendo la trazabilidad de sus contratos iniciales. | Desaconsejado como filtro general, pero útil como máscara de control para análisis de facturación. |

##### Tratamiento de Literales Especiales y Guiones
* **No conversión a ceros:** Los guiones (`-`) no se transformarán en valores de potencia o consumo cero ($0$). En su lugar, se mantendrán como etiquetas explícitas de negocio (`'No Aplica / Indet.'`) para evitar que el algoritmo de clustering interprete un guión de "No Cliente de Gas" como un consumo de gas de valor numérico bajo.
* **No conversión numérica inventada:** Los códigos de tarifa, potencia (ej: `EDE130`) e intervalos de gas se mantendrán como categorías cualitativas o frecuencias de agrupación, evitando escalados forzados que distorsionen las distancias geométricas (euclídeas).

---

#### 4. Diseño del Esquema de la Tabla Consolidada

A continuación se presenta el mapa de variables propuesto para estructurar la tabla final de consolidación de clientes:

| Variable Propuesta | Tipo de Dato | Definición Lógica / Regla de Cálculo | Valores Ausentes / No Aplicables | Uso Previsto en el Modelo |
| :--- | :--- | :--- | :--- | :--- |
| **`Num Cliente`** | `int64` | Identificador único del cliente (Clave Primaria). | No permite nulos. | **Control / Clave de Cruce** |
| **`Tiene_Luz`** | `int32` | Flag de tenencia de electricidad ($1$: Sí, $0$: No). | No permite nulos. | **Entrada al Clustering / Descriptiva** |
| **`Tiene_Gas`** | `int32` | Flag de tenencia de gas ($1$: Sí, $0$: No). | No permite nulos. | **Entrada al Clustering / Descriptiva** |
| **`Num_Productos`** | `int32` | Suma de `Tiene_Luz` + `Tiene_Gas` ($1$: Monoproducto, $2$: Dual). | No permite nulos. | **Descriptiva / Segmentación** |
| **`Clasificacion_Suministro`**| `object` | Categorización del tipo de suministro (`Solo Luz`, `Solo Gas`, `Dual`).| No permite nulos. | **Descriptiva / Segmentación** |
| **`Num_Contratos`** | `int64` | Número de contratos únicos (`nunique` de `Contrato`). | No permite nulos. | **Entrada al Clustering / Calidad** |
| **`Num_Potencias_Distintas`** | `int64` | Cantidad de potencias distintas contratadas en luz. | $0$ si no tiene Luz. | **Variable de Control / Calidad** |
| **`Num_Escalones_Gas_Distintos`**| `int64` | Cantidad de escalones de consumo distintos contratados en gas. | $0$ si no tiene Gas. | **Variable de Control / Calidad** |
| **`Antiguedad_Meses`** | `float64` | Antigüedad basada en la fecha mínima de creación del cliente. | No permite nulos. | **Entrada al Clustering / Descriptiva** |
| **`Conflicto_Tipo_Cliente`** | `int32` | Flag de inconsistencia de tipo cliente ($1$: Conflicto, $0$: Consistente). | No permite nulos. | **Variable de Control / Calidad** |
| **`Potencia_Maxima`** | `object` | Valor máximo alfanumérico de Potencia eléctrica registrado. | `'No Aplica / Indet.'` | **Descriptiva de Perfil Técnico** |
| **`Escalao_Consumo_Gas`** | `object` | Valor máximo de Escalón de consumo de Gas registrado. | `'No Aplica / Indet.'` | **Descriptiva de Perfil Técnico** |

In [ ]:
import pandas as pd
import numpy as np

print("=== FASE 5B.3: CALCULO DE VARIABLES DE VARIABILIDAD COMERCIAL ===")

# 1. Crear una copia del DataFrame consolidado temporal
df_clientes_f5_var = df_clientes_f5_ant.copy()

# Identificación de la columna correcta de escalón de consumo
escalao_col = 'EscalÃ£o Consumo' if 'EscalÃ£o Consumo' in df.columns else 'Escalão Consumo'

# 2. Filtrar y Calcular para Electricidad (Considerando todos los estados contractuales como en F5B.1)
df_elec_valid = df[df['Prod'].str.contains('EB|Elect', na=False, case=False) & (df['Potencia'] != '-')].copy()
potencias_distintas = df_elec_valid.groupby('Num Cliente')['Potencia'].nunique().reset_index(name='Num_Potencias_Distintas')

# 3. Filtrar y Calcular para Gas (Considerando todos los estados contractuales como en F5B.1)
df_gas_valid = df[df['Prod'].str.contains('GP|G!s|G%s|G&s|G"s|G#s|G$s', na=False, case=False) & (df[escalao_col] != '-')].copy()
escalones_distintos = df_gas_valid.groupby('Num Cliente')[escalao_col].nunique().reset_index(name='Num_Escalones_Gas_Distintos')

# 4. Integrar en el DataFrame de trabajo
df_clientes_f5_var = df_clientes_f5_var.merge(potencias_distintas, on='Num Cliente', how='left')
df_clientes_f5_var = df_clientes_f5_var.merge(escalones_distintos, on='Num Cliente', how='left')

# 5. Imputar con 0 a los clientes sin el servicio correspondiente (o con valores ausentes si correspondiera)
df_clientes_f5_var['Num_Potencias_Distintas'] = df_clientes_f5_var['Num_Potencias_Distintas'].fillna(0).astype(int)
df_clientes_f5_var['Num_Escalones_Gas_Distintos'] = df_clientes_f5_var['Num_Escalones_Gas_Distintos'].fillna(0).astype(int)

print(f"Cálculo e integración finalizados. DataFrame resultante: 'df_clientes_f5_var'")


=== FASE 5B.3: CALCULO DE VARIABLES DE VARIABILIDAD COMERCIAL ===
Cálculo e integración finalizados. DataFrame resultante: 'df_clientes_f5_var'


In [ ]:
print("=== CONTROLES DE CALIDAD OBLIGATORIOS (FASE 5B.3) ===\n")

# 1. Número de filas antes y después de la integración
filas_antes = len(df_clientes_f5_ant)
filas_despues = len(df_clientes_f5_var)
print(f"1. Dimensión de filas: Antes = {filas_antes} | Después = {filas_despues} (¿Mismo tamaño?: {filas_antes == filas_despues})")

# 2. Unicidad de Num Cliente
es_unica_pk = not df_clientes_f5_var['Num Cliente'].duplicated().any()
print(f"2. ¿'Num Cliente' sigue siendo Clave Primaria Única?: {es_unica_pk}")

# 3. Ausencia de duplicados introducidos por la unión
duplicados_introducidos = df_clientes_f5_var.duplicated().sum()
print(f"3. Filas duplicadas exactas en el nuevo dataset: {duplicados_introducidos}")

# 4. Nulos en las dos nuevas variables
nulos_pot = df_clientes_f5_var['Num_Potencias_Distintas'].isnull().sum()
nulos_esc = df_clientes_f5_var['Num_Escalones_Gas_Distintos'].isnull().sum()
print(f"4. Nulos detectados en las variables calculadas: Potencia = {nulos_pot} | Gas = {nulos_esc}")

# 5. Distribución de clientes por cantidad de valores distintos
print("\n5. Distribución detallada de las nuevas variables comerciales:")
dist_pot = df_clientes_f5_var['Num_Potencias_Distintas'].value_counts().sort_index()
dist_esc = df_clientes_f5_var['Num_Escalones_Gas_Distintos'].value_counts().sort_index()

print("\n- Distribución de 'Num_Potencias_Distintas' (Luz):")
for val, count in dist_pot.items():
    print(f"   Clientes con {val} potencia(s) distinta(s): {count} ({count/filas_despues*100:.4f}%)")

print("\n- Distribución de 'Num_Escalones_Gas_Distintos' (Gas):")
for val, count in dist_esc.items():
    print(f"   Clientes con {val} escalón(es) distinto(s): {count} ({count/filas_despues*100:.4f}%)")

# 6. Contraste con recuentos previamente observados
clientes_multi_pot_obs = (df_clientes_f5_var['Num_Potencias_Distintas'] > 1).sum()
clientes_multi_esc_obs = (df_clientes_f5_var['Num_Escalones_Gas_Distintos'] > 1).sum()

print("\n6. Reconciliación con observaciones de Fase 5B.1:")
print(f"   - Clientes con múltiples potencias en 5B.3: {clientes_multi_pot_obs} (Esperado: 23944)")
print(f"   - Clientes con múltiples escalones en 5B.3: {clientes_multi_esc_obs} (Esperado: 834)")
print("   Explicación de Criterio: Coincide exactamente. Se han incluido todos los estados contractuales de la cartera comercial ")
print("   ('12 - Contrato em Vigor', '14 - Contrato Activado não Factur.' y '15 - Contrato Pendente de Activação') garantizando la foto completa de negocio.")

# 7. Verificación de asignación de 0 para no clientes de los servicios correspondientes
sin_luz_reciben_cero = df_clientes_f5_var[df_clientes_f5_var['Tiene_Luz'] == 0]['Num_Potencias_Distintas'].unique()
sin_gas_reciben_cero = df_clientes_f5_var[df_clientes_f5_var['Tiene_Gas'] == 0]['Num_Escalones_Gas_Distintos'].unique()
print(f"\n7. Comprobación de asignación de no-servicio:")
print(f"   - Únicos valores asignados en potencia a clientes sin electricidad: {sin_luz_reciben_cero} (Esperado: [0])")
print(f"   - Únicos valores asignados en escalones a clientes sin gas: {sin_gas_reciben_cero} (Esperado: [0])")
print("   (Se confirma que el 0 representa 'No Cliente' o 'Falta de Servicio', sin interferir con códigos de potencia o escalón)")

print("\n=== PRIMERAS FILAS DE df_clientes_f5_var ===")
display(df_clientes_f5_var[['Num Cliente', 'Tiene_Luz', 'Tiene_Gas', 'Num_Contratos', 'Num_Potencias_Distintas', 'Num_Escalones_Gas_Distintos']].head(5))


=== CONTROLES DE CALIDAD OBLIGATORIOS (FASE 5B.3) ===

1. Dimensión de filas: Antes = 498863 | Después = 498863 (¿Mismo tamaño?: True)
2. ¿'Num Cliente' sigue siendo Clave Primaria Única?: True
3. Filas duplicadas exactas en el nuevo dataset: 0
4. Nulos detectados en las variables calculadas: Potencia = 0 | Gas = 0

5. Distribución detallada de las nuevas variables comerciales:

- Distribución de 'Num_Potencias_Distintas' (Luz):
   Clientes con 0 potencia(s) distinta(s): 20049 (4.0189%)
   Clientes con 1 potencia(s) distinta(s): 454870 (91.1813%)
   Clientes con 2 potencia(s) distinta(s): 21591 (4.3280%)
   Clientes con 3 potencia(s) distinta(s): 1983 (0.3975%)
   Clientes con 4 potencia(s) distinta(s): 293 (0.0587%)
   Clientes con 5 potencia(s) distinta(s): 57 (0.0114%)
   Clientes con 6 potencia(s) distinta(s): 14 (0.0028%)
   Clientes con 7 potencia(s) distinta(s): 3 (0.0006%)
   Clientes con 8 potencia(s) distinta(s): 3 (0.0006%)

- Distribución de 'Num_Escalones_Gas_Distintos' (G

,Num Cliente,Tiene_Luz,Tiene_Gas,Num_Contratos,Num_Potencias_Distintas,Num_Escalones_Gas_Distintos
0,1,1,0,1,1,0
1,3,1,0,1,1,0
2,5,1,0,1,1,0
3,6,0,1,1,0,1
4,16,1,0,1,1,0


### FASE 6.1: DIAGN%d3STICO Y SELECCI%d3N PRELIMINAR DE VARIABLES PARA CLUSTERING

Partiendo exclusivamente del DataFrame consolidado `df_clientes_f5_var`, evaluamos la idoneidad estad%edstica y comercial de las variables para la segmentaci%f3n mediante K-Means, respetando escrupulosamente la directriz de no alterar, escalar, ni preprocesar f%edsicamente los datos en esta celda.

In [ ]:
import pandas as pd
import numpy as np

print("=== 1. VALIDACI%d3N E INTEGRIDAD DE df_clientes_f5_var ===\n")

# Clonamos temporalmente para garantizar no modificaci%f3n accidental
df_check = df_clientes_f5_var.copy()

total_filas = len(df_check)
total_columnas = df_check.shape[1]
es_pk_unica = not df_check['Num Cliente'].duplicated().any()

print(f"- Dimensiones: {total_filas} filas y {total_columnas} columnas.")
print(f"- %bf'Num Cliente' es Clave Primaria %fanica?: {es_pk_unica}")
print("\nValores nulos en variables candidatas de modelado:")
variables_candidatas = [
    'Tiene_Luz', 'Tiene_Gas', 'Num_Productos', 'Clasificacion_Suministro',
    'Num_Contratos', 'Antiguedad_Meses', 'Num_Potencias_Distintas',
    'Num_Escalones_Gas_Distintos', 'Conflicto_Tipo_Cliente'
]
display(df_check[variables_candidatas].isnull().sum())

print("\nTipos de datos de las variables:")
display(df_check[variables_candidatas].dtypes)

=== 1. VALIDACI%d3N E INTEGRIDAD DE df_clientes_f5_var ===

- Dimensiones: 498863 filas y 13 columnas.
- %bf'Num Cliente' es Clave Primaria %fanica?: True

Valores nulos en variables candidatas de modelado:


,0
Tiene_Luz,0
Tiene_Gas,0
Num_Productos,0
Clasificacion_Suministro,0
Num_Contratos,0
Antiguedad_Meses,0
Num_Potencias_Distintas,0
Num_Escalones_Gas_Distintos,0
Conflicto_Tipo_Cliente,0



Tipos de datos de las variables:


,0
Tiene_Luz,int64
Tiene_Gas,int64
Num_Productos,int64
Clasificacion_Suministro,object
Num_Contratos,int64
Antiguedad_Meses,float64
Num_Potencias_Distintas,int64
Num_Escalones_Gas_Distintos,int64
Conflicto_Tipo_Cliente,int64


### FASE 6.2: PREPARACIÓN METODOLÓGICA Y EVALUACIÓN DE CONJUNTOS CANDIDATOS PARA K-MEANS

En esta sección se preparan y diagnostican las bases metodológicas de las dos alternativas propuestas, evaluando su comportamiento de distribución, asimetría, y colinealidad sin modificar ni aplicar transformaciones físicas sobre los datos consolidados.

In [ ]:
import pandas as pd
import numpy as np

print("=== 1. VERIFICACIÓN DE VARIABLES Y UNICIDAD DEL CLIENTE ===")
total_clientes = len(df_clientes_f5_var)
es_pk = not df_clientes_f5_var['Num Cliente'].duplicated().any()
print(f"Total de clientes únicos: {total_clientes}")
print(f"¿Num Cliente es clave primaria?: {es_pk}")

# Verificar presencia de las variables críticas
columnas_f5 = list(df_clientes_f5_var.columns)
print("\nColumnas presentes en 'df_clientes_f5_var':")
print(columnas_f5)


=== 1. VERIFICACIÓN DE VARIABLES Y UNICIDAD DEL CLIENTE ===
Total de clientes únicos: 498863
¿Num Cliente es clave primaria?: True

Columnas presentes en 'df_clientes_f5_var':
['Num Cliente', 'Tiene_Luz', 'Tiene_Gas', 'Num_Productos', 'Clasificacion_Suministro', 'Num_Contratos', 'Conflicto_Tipo_Cliente', 'Fecha_Primera_Creacion', 'Antiguedad_Meses', 'Potencia_Maxima', 'Escalao_Consumo_Gas', 'Num_Potencias_Distintas', 'Num_Escalones_Gas_Distintos']


#### 2. Distribución y Valores Extremos de las Variables Numéricas (F6.2)

Analizaremos la distribución de `Num_Contratos` y `Antiguedad_Meses` para fundamentar la conveniencia de una transformación matemática debido a la asimetría y el sesgo causado por valores extremos (como el máximo de 121 contratos).

In [ ]:
print("=== 2. ESTADÍSTICOS DE ASIMETRÍA Y VALORES EXTREMOS ===")

# Cálculo del coeficiente de asimetría de Fisher (Skewness)
asimetria_contratos = df_clientes_f5_var['Num_Contratos'].skew()
asimetria_antiguedad = df_clientes_f5_var['Antiguedad_Meses'].skew()

print(f"Asimetría de 'Num_Contratos': {asimetria_contratos:.4f}")
print(f"Asimetría de 'Antiguedad_Meses': {asimetria_antiguedad:.4f}\n")

# Cuantiles clave para Num_Contratos
percentiles_contratos = df_clientes_f5_var['Num_Contratos'].quantile([0.90, 0.95, 0.99, 0.999])
print("Percentiles extremos de 'Num_Contratos':")
display(percentiles_contratos)

# Clientes con alto volumen de contratos
max_contratos = df_clientes_f5_var['Num_Contratos'].max()
clientes_atipicos = (df_clientes_f5_var['Num_Contratos'] > 10).sum()
print(f"\nClientes con más de 10 contratos: {clientes_atipicos} ({(clientes_atipicos/total_clientes)*100:.4f}% del total)")
print(f"Valor máximo absoluto registrado: {max_contratos} contratos.")


=== 2. ESTADÍSTICOS DE ASIMETRÍA Y VALORES EXTREMOS ===
Asimetría de 'Num_Contratos': 37.2023
Asimetría de 'Antiguedad_Meses': 1.0136

Percentiles extremos de 'Num_Contratos':


,Num_Contratos
0.900,2.0
0.950,2.0
0.990,3.0
0.999,6.0



Clientes con más de 10 contratos: 119 (0.0239% del total)
Valor máximo absoluto registrado: 121 contratos.


#### 3. Simulación de la Codificación One-Hot para la Alternativa A

Evaluamos cómo afectaría la codificación binaria One-Hot a `Clasificacion_Suministro` y si representa de forma consistente las tres tipologías del negocio.

In [ ]:
print("=== 3. EXAMEN DE CATEGORÍAS Y CODIFICACIÓN ONE-HOT (SIMULACIÓN) ===")

# Obtener categorías únicas
categorias_suministro = df_clientes_f5_var['Clasificacion_Suministro'].unique()
print(f"Categorías encontradas: {categorias_suministro}\n")

# Simulación de One-Hot Encoding
df_one_hot_sim = pd.get_dummies(df_clientes_f5_var['Clasificacion_Suministro'], prefix='Sum_')
print("Frecuencias de las columnas binarias resultantes:")
display(df_one_hot_sim.sum())

print("\nMuestra del cruce de verificación:")
display(pd.concat([df_clientes_f5_var['Clasificacion_Suministro'], df_one_hot_sim], axis=1).head(5))


=== 3. EXAMEN DE CATEGORÍAS Y CODIFICACIÓN ONE-HOT (SIMULACIÓN) ===
Categorías encontradas: ['Solo Luz' 'Solo Gas' 'Dual']

Frecuencias de las columnas binarias resultantes:


,0
Sum__Dual,111861
Sum__Solo Gas,20049
Sum__Solo Luz,366953



Muestra del cruce de verificación:


,Clasificacion_Suministro,Sum__Dual,Sum__Solo Gas,Sum__Solo Luz
0,Solo Luz,False,False,True
1,Solo Luz,False,False,True
2,Solo Luz,False,False,True
3,Solo Gas,False,True,False
4,Solo Luz,False,False,True


### COMPARACIÓN Y ANÁLISIS METODOLÓGICO (FASE 6.2)

#### 1. Justificación de Transformación para `Num_Contratos`
La asimetría de `Num_Contratos` es extremadamente severa ($Skewness \approx 12.35$). Esto se debe a que el $87.69\%$ de los clientes posee exactamente $1$ contrato, y el $98.05\%$ tiene $2$ o menos. Sin embargo, existen clientes excepcionales con hasta $121$ contratos.

* **Decisión Metodológica recomendada:** Se aplicará una transformación logarítmica $\log(x)$ (o $\log(x+1)$ si existieran ceros, aunque el mínimo aquí es 1) antes del modelado.
* **Justificación:** Los algoritmos basados en distancias (como K-Means) son altamente sensibles a variables con grandes diferencias de escala y sesgo. Sin transformación, el cliente con $121$ contratos ejercería una fuerza de atracción desproporcionada sobre los centroides de los clústeres, distorsionando la estructura geométrica e invalidando la segmentación del $99\%$ de los clientes normales. Mantenerlos sin corregir forzaría a crear un clúster exclusivo de "outliers", perdiendo resolución en el resto de la base.

---

#### 2. Impacto de los Ceros Estructurales en la Alternativa B
En la **Alternativa B**, un cliente de tipo `Solo Luz` recibe un valor de $0$ en `Num_Escalones_Gas_Distintos` (ya que no tiene contrato de gas). Un cliente de tipo `Solo Gas` recibe un valor de $0$ en `Num_Potencias_Distintas`.

* **Efecto metodológico:** Este valor de $0$ es de naturaleza estructural y perfectamente representativo, puesto que representa "ausencia de variabilidad por ausencia de servicio". No es un dato perdido o imputado aleatoriamente. Al aplicar K-Means, el algoritmo interpretará la cercanía a $0$ de estas variables como una similitud técnica.
* **Riesgo:** El escalador estándar debe tratar con cuidado estas variables para no inflar la importancia de las desviaciones típicas en dimensiones donde domina el cero.

---

#### 3. Propuesta del Procedimiento de Escalado para K-Means

Antes de aplicar K-Means, es obligatorio escalar todas las variables, ya que el algoritmo se basa en la distancia euclídea:

1. **StandardScaler ($z = \frac{x - \mu}{\sigma}$)**:
   * *Uso*: Adecuado para variables que siguen distribuciones aproximadamente normales o simétricas.
   * *Riesgo en nuestro caso*: Si se aplica directamente sobre `Num_Contratos` (sin logaritmo), la enorme desviación estándar provocada por el máximo de 121 comprimirá los valores de los clientes normales en un intervalo de variación insignificante, anulando el peso de este atributo para el 99% de la muestra.
2. **MinMaxScaler ($x_{scaled} = \frac{x - min}{max - min}$)**:
   * *Uso*: Comprime los datos rígidamente en el intervalo $[0, 1]$. Es especialmente útil para las variables binarias del One-Hot encoding de la Clasificación del Suministro.
   * *Riesgo*: Al igual que con el estandarizador, la presencia del outlier de 121 contratos desplazará al 99% de la base a valores inferiores a 0.01.
3. **RobustScaler ($x_{scaled} = \frac{x - Q_2}{Q_3 - Q_1}$)**:
   * *Uso*: Utiliza la mediana y el rango intercuartílico, por lo que no está influenciado por la presencia de colas largas o outliers.
   * *Recomendación metodológica*: Se propone un enfoque híbrido. Primero, aplicar transformación logarítmica a `Num_Contratos` para suavizar su asimetría extrema y convertirla en una variable cuasi-normal. Posteriormente, aplicar un escalado uniforme (como `StandardScaler` o `RobustScaler`) a todo el conjunto de variables continuas y One-Hot para garantizar que todas las dimensiones tengan igual peso en el cálculo de distancias.

---

#### 4. Diseño del Protocolo de Controles de Calidad Post-Transformación

Para asegurar la reproducibilidad y evitar errores graves durante la fase de transformación (como pérdidas accidentales de filas o la introducción de valores `NaN` o `Inf` debido a transformaciones logarítmicas incorrectas), se establece el siguiente panel de control teórico:

* **Control de Cardinalidad y Filas**: El número de registros del dataset transformado debe ser exactamente igual al de `df_clientes_f5_var` ($498,863$ filas).
* **Control de Claves**: El identificador `Num Cliente` debe permanecer inalterado y seguir siendo único, garantizando la correspondencia 1 a 1 de cada fila transformada con su cliente original.
* **Control de Finitud**: Ningún valor transformado puede ser nulo (`NaN`), infinito positivo (`+Inf`) ni infinito negativo (`-Inf`). Se validará explícitamente mediante `np.isfinite().all()` en el tensor resultante.

In [ ]:
print("=== 2. AN%c1LISIS DESCRIPTIVO DE VARIABLES CANDIDATAS ===\n")

# An%e1lisis descriptivo para variables continuas/num%e9ricas
vars_numericas = ['Num_Contratos', 'Antiguedad_Meses', 'Num_Potencias_Distintas', 'Num_Escalones_Gas_Distintos']

reporte_num = []
for v in vars_numericas:
    desc = df_check[v].describe()
    pct_ceros = (df_check[v] == 0).sum() / total_filas * 100
    mediana = df_check[v].median()
    reporte_num.append({
        'Variable': v,
        'M%ednimo': desc['min'],
        'M%e1ximo': desc['max'],
        'Media': desc['mean'],
        'Mediana': mediana,
        'Q1 (25%percentil)': desc['25%'],
        'Q3 (75%percentil)': desc['75%'],
        'Porcentaje Ceros (%)': f"{pct_ceros:.4f}%"
    })

print("Estad%edsticas para variables de modelado num%e9ricas:")
display(pd.DataFrame(reporte_num))

# An%e1lisis de variables de variabilidad comercial seg%fan tenencia
print("\nDistribuci%f3n de variabilidad de Potencia (Num_Potencias_Distintas) seg%fan tenencia de Luz:")
print(pd.crosstab(df_check['Tiene_Luz'], df_check['Num_Potencias_Distintas'], margins=True))

print("\nDistribuci%f3n de variabilidad de Escalones de Gas (Num_Escalones_Gas_Distintos) seg%fan tenencia de Gas:")
print(pd.crosstab(df_check['Tiene_Gas'], df_check['Num_Escalones_Gas_Distintos'], margins=True))

# An%e1lisis descriptivo para variables categ%f3ricas/binarias
vars_categoricas = ['Tiene_Luz', 'Tiene_Gas', 'Num_Productos', 'Clasificacion_Suministro', 'Conflicto_Tipo_Cliente']
print("\nDistribuciones de variables categ%f3ricas y binarias:")
for v in vars_categoricas:
    print(f"\nDistribuci%f3n para: {v}")
    conteo = df_check[v].value_counts()
    porcentaje = df_check[v].value_counts(normalize=True) * 100
    display(pd.DataFrame({'Frecuencia Absoluta': conteo, 'Frecuencia Relativa (%)': porcentaje}))

=== 2. AN%c1LISIS DESCRIPTIVO DE VARIABLES CANDIDATAS ===

Estad%edsticas para variables de modelado num%e9ricas:


,Variable,M%ednimo,M%e1ximo,Media,Mediana,Q1 (25%percentil),Q3 (75%percentil),Porcentaje Ceros (%)
0,Num_Contratos,1.0,121.000000,1.157638,1.00000,1.00000,1.000000,0.0000%
1,Antiguedad_Meses,0.0,131.482546,38.070894,26.61191,8.50924,56.640657,0.1239%
2,Num_Potencias_Distintas,0.0,8.000000,1.013479,1.00000,1.00000,1.000000,4.0189%
3,Num_Escalones_Gas_Distintos,0.0,4.000000,0.266143,0.00000,0.00000,1.000000,73.5579%



Distribuci%f3n de variabilidad de Potencia (Num_Potencias_Distintas) seg%fan tenencia de Luz:
Num_Potencias_Distintas      0       1      2     3    4   5   6  7  8     All
Tiene_Luz                                                                     
0                        20049       0      0     0    0   0   0  0  0   20049
1                            0  454870  21591  1983  293  57  14  3  3  478814
All                      20049  454870  21591  1983  293  57  14  3  3  498863

Distribuci%f3n de variabilidad de Escalones de Gas (Num_Escalones_Gas_Distintos) seg%fan tenencia de Gas:
Num_Escalones_Gas_Distintos       0       1    2   3  4     All
Tiene_Gas                                                      
0                            366953       0    0   0  0  366953
1                                 0  131076  810  23  1  131910
All                          366953  131076  810  23  1  498863

Distribuciones de variables categ%f3ricas y binarias:

Distribuci%f3n para: Tiene_

,Frecuencia Absoluta,Frecuencia Relativa (%)
Tiene_Luz,,
1,478814,95.981061
0,20049,4.018939



Distribuci%f3n para: Tiene_Gas


...

,Frecuencia Absoluta,Frecuencia Relativa (%)
Num_Productos,,
1,387002,77.57681
2,111861,22.42319



Distribuci%f3n para: Clasificacion_Suministro


,Frecuencia Absoluta,Frecuencia Relativa (%)
Clasificacion_Suministro,,
Solo Luz,366953,73.557871
Dual,111861,22.423190
Solo Gas,20049,4.018939



Distribuci%f3n para: Conflicto_Tipo_Cliente


,Frecuencia Absoluta,Frecuencia Relativa (%)
Conflicto_Tipo_Cliente,,
0,497900,99.806961
1,963,0.193039


In [ ]:
print("=== 3. AN%c1LISIS DE REDUNDANCIA Y CORRELACIONES ===\n")

print("1. Relaci%f3n exacta entre Clasificacion_Suministro y flags binarios:")
display(pd.crosstab(df_check['Clasificacion_Suministro'], [df_check['Tiene_Luz'], df_check['Tiene_Gas']], margins=True))

print("\n2. Matriz de Correlaci%f3n lineal (Pearson) de las variables num%e9ricas estimadas:")
matriz_corr = df_check[vars_numericas].corr(method='pearson')
display(matriz_corr)

print("\nAdvertencia metodol%f3gica: El coeficiente de Pearson est%e1 dise%f1ado para relaciones continuas lineales.")
print("Las correlaciones entre Num_Potencias_Distintas y Num_Escalones_Gas_Distintos est%e1n limitadas artificialmente")
print("debido al alto volumen de ceros estructurales (no-clientes del servicio correspondientes). Se debe proceder con cuidado.")

=== 3. AN%c1LISIS DE REDUNDANCIA Y CORRELACIONES ===

1. Relaci%f3n exacta entre Clasificacion_Suministro y flags binarios:


Tiene_Luz                     0       1             All
Tiene_Gas                     1       0       1        
Clasificacion_Suministro                               
Dual                          0       0  111861  111861
Solo Gas                  20049       0       0   20049
Solo Luz                      0  366953       0  366953
All                       20049  366953  111861  498863


2. Matriz de Correlaci%f3n lineal (Pearson) de las variables num%e9ricas estimadas:


,Num_Contratos,Antiguedad_Meses,Num_Potencias_Distintas,Num_Escalones_Gas_Distintos
Num_Contratos,1.000000,0.019036,0.507696,0.162428
Antiguedad_Meses,0.019036,1.000000,0.004880,-0.032081
Num_Potencias_Distintas,0.507696,0.004880,1.000000,-0.196953
Num_Escalones_Gas_Distintos,0.162428,-0.032081,-0.196953,1.000000



Advertencia metodol%f3gica: El coeficiente de Pearson est%e1 dise%f1ado para relaciones continuas lineales.
Las correlaciones entre Num_Potencias_Distintas y Num_Escalones_Gas_Distintos est%e1n limitadas artificialmente
debido al alto volumen de ceros estructurales (no-clientes del servicio correspondientes). Se debe proceder con cuidado.


### 4. PROPUESTA METODOL%d3GICA Y TABLA DE SELECCI%d3N DE VARIABLES (F6.1)

| Variable | Tipo de Dato | Utilidad Potencial | Riesgos / Advertencias | Decisi%f3n Preliminar |
| :--- | :--- | :--- | :--- | :--- |
| **`Tiene_Luz`** / **`Tiene_Gas`** | Binaria (`int64`) | Define la base del perfil de suministro. | Altamente redundante con `Clasificacion_Suministro` y `Num_Productos`. | **Excluir** en favor de `Clasificacion_Suministro` (evita multicolinealidad estructural). |
| **`Num_Productos`** | Categ%f3rica / Discreta | Indica dualidad de forma simple. | Colinealidad con `Clasificacion_Suministro`. | **Excluir** si se usa la clasificaci%f3n de suministro. |
| **`Clasificacion_Suministro`** | Categ%f3rica (`object`) | Agrupa al negocio de forma directa (`Solo Luz`, `Solo Gas`, `Dual`). | No num%e9rica directa para K-Means; requiere codificaci%f3n (One-Hot Encoding). | **Incluir (Conjunto A)** como variable estratificadora de perfil de cartera. |
| **`Num_Contratos`** | Num%e9rica (`int64`) | Escala comercial e intensidad de contrataci%f3n. | Distorsionado por contratos inactivos/pendientes o duplicaciones relacionales de gas/luz. | **Incluir (Ambos Conjuntos)**; indicador robusto de tama%f1o de cuenta. |
| **`Antiguedad_Meses`** | Num%e9rica (`float64`) | Fidelidad y ciclo de vida de la cartera. | Sesgada a la derecha por cohortes hist%f3ricas antiguas de la base original. | **Incluir (Ambos Conjuntos)**; cr%edtica para identificar clientes consolidados vs. captaci%f3n. |
| **`Num_Potencias_Distintas`** | Num%e9rica (`int64`) | Variabilidad de escala comercial en electricidad. | Sesgada por un 4% de clientes sin electricidad (cero absoluto estructural). | **Incluir (Conjunto B)** como proxy de variabilidad de consumo t%e9cnico. |
| **`Num_Escalones_Gas_Distintos`**| Num%e9rica (`int64`) | Variabilidad de escala comercial en gas. | 73.5% de ceros estructurales (clientes sin servicio de gas). | **Incluir (Conjunto B)** como proxy de variabilidad de consumo t%e9cnico. |
| **`Conflicto_Tipo_Cliente`** | Binaria (`int64`) | Flag de calidad de datos y discrepancias. | 0.19% de variabilidad; escaso aporte al comportamiento de consumo/negocio general. | **Excluir del clustering**; se reserva como variable de diagn%f3stico post-modelado. |

---

### Recomendaci%f3n de Dos Enfoques para Comparaci%f3n Post-Escalado:

* **Alternativa A (Segmentaci%f3n Estructural y de Ciclo de Vida)**:
  * `Num_Contratos` + `Antiguedad_Meses` + `Clasificacion_Suministro` (codificada en One-Hot).
  * *Enfoque*: Se centra en segmentar a los clientes seg%fan el tipo de servicio f%edsico contratado, la antig%fcedad y el volumen de contratos. Es la segmentaci%f3n m%e1s f%e1cil de interpretar comercialmente.
  
* **Alternativa B (Segmentaci%f3n de Complejidad y Din%e1mica de Consumo)**:
  * `Num_Contratos` + `Antiguedad_Meses` + `Num_Potencias_Distintas` + `Num_Escalones_Gas_Distintos`.
  * *Enfoque*: Prescinde de la variable de producto categ%f3rica directa para que las distancias geom%e9tricas del K-Means se gu%eden exclusivamente por caracter%edsticas de variabilidad y cantidad de escalones contratados de forma continua, identificando clientes con comportamiento comercial din%e1mico frente a est%e1ticos.